0 Install all system & Python dependencies

In [ ]:
%%bash
# ─── Linux packages ──────────────────────────────────────────────────────────
apt-get -y update && \
apt-get -y install \
    build-essential aria2 pigz bcftools tabix curl \
    libbz2-dev liblzma-dev libcurl4-openssl-dev libhts-dev zlib1g-dev

# ─── Python libraries ────────────────────────────────────────────────────────
pip install --quiet --upgrade pip
pip install --quiet \
    pandas pyarrow numpy scipy \
    gget pysradb mygene==3.2.2 \
    cyvcf2 pybigwig tqdm


1 Define project folders & helper globals

In [ ]:
# ── mount your Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── pick a folder INSIDE Drive to hold everything ───────────────────────
import pathlib, shutil, os

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/dmd_dmd_graph')  # change if you like
RAW   = DRIVE_ROOT/'raw'
PROC  = DRIVE_ROOT/'proc'
DATA  = DRIVE_ROOT                 # keep index in the same root

# make the directories if they don’t exist
for p in (RAW, PROC):
    p.mkdir(parents=True, exist_ok=True)

print("🔗  Project folders:")
for p in (RAW, PROC):
    print(" •", p)


# Narrow slice of genes implicated in neuromuscular biology
NEURO_MUSCLE_GENES = [
    # hand-curated quick start (extend as needed)
    "DMD","UTRN","LTBP4","SPP1","CD40","JAG1",
    "ANKRD2","FKRP","SGCA","CAV3","DYSF","LMNA"
]


In [ ]:
%%bash
# ── 1  Remove the wrong “salmon” Python package ─────────────────────────────
pip uninstall -y salmon || true          # ignore if not present

# ── 2  Install the real Salmon 1.10.2 binary (static, no conda needed) ──────
SALMON_VER=1.10.2
curl -L -o salmon.tar.gz \
  https://github.com/COMBINE-lab/salmon/releases/download/v${SALMON_VER}/salmon-${SALMON_VER}_linux_x86_64.tar.gz
tar -xzf salmon.tar.gz
mv salmon-${SALMON_VER}_linux_x86_64/bin/salmon /usr/local/bin/
chmod +x /usr/local/bin/salmon
salmon -h | head -n 3   # sanity check: should print Salmon banner

# ── 3  Install SRA-Toolkit (so we have fasterq-dump) ────────────────────────
apt-get -y install sra-toolkit


In [ ]:
%%bash
apt-get -y update
apt-get -y install salmon   # pulls salmon-1.6.x

echo "🏃  Salmon installed → $(salmon --version)"


In [ ]:
# 2) Install SRA-toolkit + pigz
!apt-get update -qq
!apt-get install -y sra-toolkit pigz

# 3) Download Salmon v1.4.0 (follow redirects!) and unpack
!wget -qO salmon.tar.gz -L \
    https://github.com/COMBINE-lab/salmon/releases/download/v1.4.0/salmon_linux_x86_64.tar.gz
!tar -xzf salmon.tar.gz

# 4) Copy the binary into /usr/local/bin so it's on your PATH
!cp salmon_linux_x86_64/bin/salmon /usr/local/bin/

# 5) Verify it’s working
!salmon --version

**2 Block A — Build the gene-level expression matrix**

---



In [ ]:
from pathlib import Path
from pysradb import SRAweb
import pandas as pd

# connect to SRA
sra = SRAweb()

# helper regex to drop all single‐cell runs
drop_sc = r"single cell|scrna"

# ── 1) DMD bulk runs ─────────────────────────────────────────────────────────
dmd_query = (
    "Duchenne muscular dystrophy AND muscle "
    "AND Homo sapiens[Organism] AND RNA-Seq[Strategy]"
)
dmd_df = (
    sra.search_sra(dmd_query)
       .query('library_strategy == "RNA-Seq"')
       .query('library_selection.str.lower() != "cage"', engine="python")
       .query(f'~library_source.str.lower().str.contains("{drop_sc}")', engine="python")
       .query(f'~experiment_title.str.lower().str.contains("{drop_sc}")', engine="python")
)
dmd_runs = (
    dmd_df
      .sort_values("total_spots", ascending=True)
      .head(3)
      .assign(phenotype="DMD")
)

# ── 2) healthy bulk controls ─────────────────────────────────────────────────
ctrl_query = (
    "(healthy OR normal OR control OR WT OR wild-type) AND muscle "
    "AND Homo sapiens[Organism] AND RNA-Seq[Strategy]"
)
ctrl_df = (
    sra.search_sra(ctrl_query)
       .query('library_strategy == "RNA-Seq"')
       .query('library_selection.str.lower() != "cage"', engine="python")
       .query(f'~library_source.str.lower().str.contains("{drop_sc}")', engine="python")
       .query(f'~experiment_title.str.lower().str.contains("{drop_sc}")', engine="python")
       .query('~study_title.str.lower().str.contains("dmd|duchenne")', engine="python")
)
ctrl_runs = (
    ctrl_df
      .sort_values("total_spots", ascending=True)
      .head(80)
      .assign(phenotype="CTRL")
)

# ── 3) Combine & save ──────────────────────────────────────────────────────
runs = pd.concat([dmd_runs, ctrl_runs], ignore_index=True).reset_index(drop=True)
# drop rows 2,3,4 and reset the integer index
# specify the integer‐positions you want to keep
keep_idxs = [0, 1, 2, 7, 30, 50, 69]

# select them and reset the index
runs = runs.iloc[keep_idxs].reset_index(drop=True)


runs.to_csv(RAW / "bulk_rna.csv", index=False)

print("✅ Saved runs by phenotype:", runs["phenotype"].value_counts().to_dict())
display(runs[[
    "run_accession","phenotype",
    "experiment_title","sample_title","library_source","total_spots"
]])


In [ ]:
from pathlib import Path
import subprocess

# 1) Set your data folder
DATA_DIR = Path("/content/drive/MyDrive/dmd_dmd_graph/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 2) URLs and target paths
URL = (
    "ftp://ftp.ebi.ac.uk/pub/databases/gencode/"
    "Gencode_human/release_39/gencode.v39.transcripts.fa.gz"
)
gz_path = DATA_DIR / "gencode.v39.transcripts.fa.gz"
fa_path = DATA_DIR / "transcripts.fa"

# 3) Download if not already present
if not fa_path.exists():
    print("⏬ Downloading GENCODE v39 transcripts…")
    subprocess.run([
        "wget", "-q", "-O", str(gz_path), URL
    ], check=True)

    # 4) Unzip
    print("🗜️  Uncompressing…")
    subprocess.run([
        "gunzip", "-f", str(gz_path)
    ], check=True)

    # 5) Rename to transcripts.fa
    orig_fa = DATA_DIR / "gencode.v39.transcripts.fa"
    orig_fa.rename(fa_path)
    print(f"✅ Saved transcriptome to {fa_path}")
else:
    print(f"✅ {fa_path} already exists, skipping download.")


In [ ]:
from pathlib import Path
import subprocess

# Define data and index paths
DATA_DIR = Path("/content/drive/MyDrive/dmd_dmd_graph/data")
TRANSCRIPTS_FA = DATA_DIR / "transcripts.fa"
INDEX_DIR = DATA_DIR / "salmon_index_GRCh38"

# 1) Ensure the parent directories exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 2) Now make the index directory
INDEX_DIR.mkdir(parents=True, exist_ok=True)

# 3) Build the Salmon index
print(f"⚙️  Building Salmon index at: {INDEX_DIR}")
subprocess.run([
    "salmon", "index",
    "-t", str(TRANSCRIPTS_FA),
    "-i", str(INDEX_DIR),
    "--threads", "4"
], check=True)

# 4) Confirm creation of versionInfo.json
version_file = INDEX_DIR / "versionInfo.json"
if version_file.exists():
    print(f"✅ Salmon index built successfully! versionInfo.json located at:\n  {version_file}")
else:
    print("❌ Salmon index build failed: versionInfo.json not found.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
# Base project path on Drive
BASE = "/content/drive/MyDrive/dmd_dmd_graph"
RAW      = os.path.join(BASE, "raw")
FASTQ    = os.path.join(RAW, "fastq")
QUANT    = os.path.join(RAW, "quant")
DATA     = os.path.join(BASE, "data")
INDEX    = os.path.join(DATA, "salmon_index_GRCh38")

# Make sure folders exist
os.makedirs(FASTQ, exist_ok=True)
os.makedirs(QUANT, exist_ok=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path
import subprocess, shutil, glob, os

# Load runs table
runs = pd.read_csv("/content/drive/MyDrive/dmd_dmd_graph/raw/bulk_rna.csv")

BASE      = Path("/content/drive/MyDrive/dmd_dmd_graph")
FASTQ_DIR = BASE / "raw" / "fastq";  FASTQ_DIR.mkdir(exist_ok=True, parents=True)
QUANT_DIR = BASE / "raw" / "quant";  QUANT_DIR.mkdir(exist_ok=True, parents=True)
INDEX_DIR = BASE / "data" / "salmon_index_GRCh38"
assert (INDEX_DIR / "versionInfo.json").exists(), "Build Salmon index first."

def run(cmd):
    print("→", " ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
# Change this to the SRR accession you want to process:
sr = runs["run_accession"].iloc[6]
print("🧪 Will quantify:", sr)


In [ ]:
outdir = QUANT_DIR / f"{sr}_quant"
if outdir.exists():
    print(f"✔️ {sr} already quantified; skipping.")
else:
    run(["prefetch", sr, "-O", str(FASTQ_DIR)])
    run([
        "fasterq-dump","--split-files",
        "-O", str(FASTQ_DIR),
        str(FASTQ_DIR/sr)
    ])
    for fq in glob.glob(f"{FASTQ_DIR}/{sr}*.fastq"):
        run(["pigz","-p","4",fq])
    fq1 = FASTQ_DIR/f"{sr}_1.fastq.gz"
    fq2 = FASTQ_DIR/f"{sr}_2.fastq.gz"
    single = FASTQ_DIR/f"{sr}.fastq.gz"
    if fq1.exists() and fq2.exists():
        cmd = ["salmon","quant","-i",str(INDEX_DIR),"-l","A",
               "-1",str(fq1),"-2",str(fq2),"-o",str(outdir),"--validateMappings"]
    else:
        cmd = ["salmon","quant","-i",str(INDEX_DIR),"-l","A",
               "-r",str(single),"-o",str(outdir),"--validateMappings"]
    run(cmd)
    shutil.rmtree(FASTQ_DIR/sr, ignore_errors=True)
    for f in FASTQ_DIR.glob(f"{sr}*"):
        os.remove(f)
    print(f"✅ Finished {sr}, only quant folder remains.")


2.1 Expression Matrix Development

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell: Download GENCODE v39 GTF and build tx2gene.tsv
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import gzip
import requests
from pathlib import Path

# 1) Define where to save the downloaded GTF and tx2gene mapping
DATA_DIR = Path("/content/drive/MyDrive/dmd_dmd_graph/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

gtf_gz_path   = DATA_DIR / "gencode.v39.annotation.gtf.gz"
tx2gene_path  = DATA_DIR / "gencode_v39_tx2gene.tsv"

# 2) Download GENCODE v39 GTF (if it does not already exist)
if not gtf_gz_path.exists():
    print("→ Downloading GENCODE v39 GTF (this may take a minute)…")
    url = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_39/gencode.v39.annotation.gtf.gz"
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(gtf_gz_path, "wb") as f_out:
        for chunk in r.iter_content(chunk_size=1<<20):
            f_out.write(chunk)
    print("✅ Download complete.")

# 3) Parse the GTF to extract transcript_id → gene_id mapping
#    We'll only process lines where column 3 == "transcript" (to avoid duplicates).
print("→ Extracting transcript_id → gene_id from GTF…")

tx2gene = {}  # dictionary: { transcript_id : gene_id }

with gzip.open(gtf_gz_path, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.split("\t")
        if len(cols) < 9:
            continue
        feature_type = cols[2]
        if feature_type != "transcript":
            continue

        # The 9th column is the attribute column. It looks like:
        #   gene_id "ENSG00000123456"; transcript_id "ENST00000123456"; ...
        attr_field = cols[8]
        # We can split on ';' and strip each piece
        attr_parts = [x.strip() for x in attr_field.split(";") if x.strip()]
        gene_id = None
        tx_id   = None
        for part in attr_parts:
            if part.startswith('gene_id '):
                # part might look like: gene_id "ENSG00000123456"
                gene_id = part.split()[1].strip('"')
            elif part.startswith('transcript_id '):
                tx_id = part.split()[1].strip('"')
            if gene_id and tx_id:
                break

        if (gene_id is not None) and (tx_id is not None):
            tx2gene[tx_id] = gene_id

print(f"ℹ️  Found {len(tx2gene):,} transcript → gene mappings.")

# 4) Write out the mapping to a two‐column TSV (no header)
with open(tx2gene_path, "w") as out:
    for tx, gene in tx2gene.items():
        out.write(f"{tx}\t{gene}\n")

print("✅ Wrote tx2gene mapping to:", tx2gene_path)


Quant to TPM (Transcript)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path

# ── 1) Define project directories ────────────────────────────────────────────
BASE      = Path("/content/drive/MyDrive/dmd_dmd_graph")
QUANT_DIR = BASE / "raw" / "quant"           # where your Salmon outputs live
PROC      = BASE / "proc"                    # where you'll save processed files
PROC.mkdir(exist_ok=True, parents=True)

# ── 2) Load your runs table ─────────────────────────────────────────────────
runs = pd.read_csv(BASE / "raw" / "bulk_rna.csv")

# ── 3) Helper to read one Salmon quant.sf ───────────────────────────────────
def load_tpm(qdir: Path) -> pd.DataFrame:
    sf = qdir / "quant.sf"
    df = pd.read_csv(sf, sep="\t", usecols=["Name", "TPM"])
    df = df.rename(columns={"TPM": qdir.name})
    return df.set_index("Name")

# ── 4) Iterate through each run_accession, gather TPM tables ────────────────
tpm_tables = []
missing = []
for sr in runs["run_accession"]:
    qdir = QUANT_DIR / f"{sr}_quant"
    sf = qdir / "quant.sf"
    if sf.exists():
        tpm_tables.append(load_tpm(qdir))
    else:
        missing.append(sr)

print(f"⚠️  missing quant.sf for runs: {missing}")

if not tpm_tables:
    raise RuntimeError("No quant.sf files found! Check that Salmon quantification produced outputs in raw/quant.")

# ── 5) Concatenate into one gene×samples TPM matrix ─────────────────────────
expr = pd.concat(tpm_tables, axis=1).fillna(0)
expr.index.name = "gene"

# ── 6) Save to disk as expression_final.parquet ─────────────────────────────
expr.to_parquet(PROC / "expression_final.parquet", compression="snappy")

print("✅  Final expression matrix saved →", (PROC / "expression_final.parquet"))
print("✅  Final expression matrix shape:", expr.shape)


Gene level expression matrix

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell: Collapse Salmon’s transcript‐level TPM → gene‐level TPM (with gene symbols)
# ──────────────────────────────────────────────────────────────────────────────

from pathlib import Path
import pandas as pd
import gzip
import re
import requests
import shutil

# ──────────────────────────────────────────────────────────────────────────────
# 0) Paths (adjust if needed)
# ──────────────────────────────────────────────────────────────────────────────
BASE            = Path("/content/drive/MyDrive/dmd_dmd_graph")
DATA_DIR        = BASE / "data"
PROC_DIR        = BASE / "proc"

TX_PARQUET      = PROC_DIR / "expression_final.parquet"      # Salmon transcript‐level TPM
TX2GENE_TSV     = DATA_DIR / "gencode_v39_tx2gene.tsv"       # GENCODE v39 tx2gene map
GTF_GZ_URL      = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_39/gencode.v39.annotation.gtf.gz"
GTF_PATH        = DATA_DIR / "gencode.v39.annotation.gtf"
GENE_PARQUET    = PROC_DIR / "expression_gene_tpm_named.parquet"

# Make sure directories exist
DATA_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
# 0.1) If GTF is missing, download & unzip it via requests
# ──────────────────────────────────────────────────────────────────────────────
if not GTF_PATH.exists():
    print("→ GTF not found. Downloading GENCODE v39 GTF (gzipped) and unzipping...")
    gz_path = DATA_DIR / "gencode.v39.annotation.gtf.gz"

    # Download the .gtf.gz
    with requests.get(GTF_GZ_URL, stream=True) as r:
        r.raise_for_status()
        with open(gz_path, "wb") as f_out:
            for chunk in r.iter_content(chunk_size=1_048_576):
                f_out.write(chunk)
    print("→ Download complete. Unzipping now…")

    # Unzip using gzip module
    with gzip.open(gz_path, "rb") as f_in:
        with open(GTF_PATH, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

    # Remove the .gz file to save space
    gz_path.unlink()
    print("✅ Downloaded and extracted gencode.v39.annotation.gtf\n")
else:
    print("✅ GTF already present at:", GTF_PATH, "\n")

# ──────────────────────────────────────────────────────────────────────────────
# 1) Load transcript‐level TPM matrix
# ──────────────────────────────────────────────────────────────────────────────
expr_tx = pd.read_parquet(TX_PARQUET)
print(f"Loaded transcript‐level TPM matrix: {expr_tx.shape}")
print("Example raw index entries (first 3):")
print(expr_tx.index[:3].tolist(), "\n")

# ──────────────────────────────────────────────────────────────────────────────
# 2) “Clean” Salmon transcript IDs:
#    a) Keep only the portion before the first ‘|’
#    b) Drop the “.version” (everything after the first ‘.’)
# ──────────────────────────────────────────────────────────────────────────────
tx_ids = expr_tx.index.to_series().astype(str)

# 2a) Split on '|' → keep piece[0], e.g. “ENST00000456328.2”
tx_ids = tx_ids.str.split(r"\|").str[0]

# 2b) Split on '.' → keep piece[0], e.g. “ENST00000456328”
tx_ids = tx_ids.str.split(r"\.").str[0]

# Replace the DataFrame’s index with these cleaned IDs
expr_tx.index = tx_ids
print("After cleaning, example index entries:")
print(expr_tx.index[:3].tolist(), "\n")

# ──────────────────────────────────────────────────────────────────────────────
# 3) Collapse duplicates (if two transcript‐versions became identical after cleaning)
# ──────────────────────────────────────────────────────────────────────────────
expr_tx = expr_tx.groupby(expr_tx.index).sum()
print(f"After collapsing duplicates, expr_tx shape = {expr_tx.shape}\n")

# ──────────────────────────────────────────────────────────────────────────────
# 4) Load the tx→gene mapping (GENCODE v39) and drop “.version” there, too
# ──────────────────────────────────────────────────────────────────────────────
tx2gene = pd.read_csv(
    TX2GENE_TSV,
    sep="\t",
    header=None,
    names=["transcript_id", "gene_id"]
)

# Strip version from each transcript_id
tx2gene["transcript_id"] = tx2gene["transcript_id"].astype(str).str.split(r"\.").str[0]
tx2gene = tx2gene.set_index("transcript_id")

print(f"Loaded tx2gene mapping: {len(tx2gene):,} rows")
print("Example tx2gene (cleaned) first 5 rows:")
print(tx2gene.head(), "\n")

# ──────────────────────────────────────────────────────────────────────────────
# 5) Restrict to transcripts present in both expr_tx and tx2gene
# ──────────────────────────────────────────────────────────────────────────────
common_txs = expr_tx.index.intersection(tx2gene.index)
print(f"Transcripts in common after cleaning: {len(common_txs):,}\n")

expr_tx = expr_tx.loc[common_txs].copy()

# ──────────────────────────────────────────────────────────────────────────────
# 6) Map transcript → gene using a plain dict (avoids InvalidIndexError)
# ──────────────────────────────────────────────────────────────────────────────
tx_to_gene_dict = tx2gene["gene_id"].to_dict()
expr_tx["gene_id"] = expr_tx.index.map(tx_to_gene_dict)

# Drop any transcripts that failed to map
n_missing = expr_tx["gene_id"].isna().sum()
if n_missing > 0:
    print(f"⚠️  Warning: {n_missing} transcripts did not map to any gene_id; dropping them.")
    expr_tx = expr_tx.dropna(subset=["gene_id"])

# ──────────────────────────────────────────────────────────────────────────────
# 7) Collapse to gene‐level TPM by summing all transcript TPMs per gene
# ──────────────────────────────────────────────────────────────────────────────
expr_gene = (
    expr_tx
      .set_index("gene_id", append=False)  # now index is gene_id
      .groupby(level="gene_id")
      .sum()
)
expr_gene.index.name = "gene_id"
print(f"Collapsed to gene‐level TPM matrix: {expr_gene.shape}\n")

# ──────────────────────────────────────────────────────────────────────────────
# 8) Parse GTF to build gene_id → gene_name map
# ──────────────────────────────────────────────────────────────────────────────
gene2name = {}
with open(GTF_PATH, "r") as gtf_handle:
    for line in gtf_handle:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if cols[2] != "gene":
            continue
        attrs = cols[8]
        # Extract gene_id and gene_name
        m_id   = re.search(r'gene_id "([^"]+)"', attrs)
        m_name = re.search(r'gene_name "([^"]+)"', attrs)
        if m_id and m_name:
            gid   = m_id.group(1)
            gname = m_name.group(1)
            gene2name[gid] = gname

print(f"Built gene2name map for {len(gene2name)} genes")

# ──────────────────────────────────────────────────────────────────────────────
# 9) Map gene_id index → gene_name, fill missing with original gene_id
# ──────────────────────────────────────────────────────────────────────────────
expr_gene["gene_name"] = expr_gene.index.map(gene2name)

n_missing_symbols = expr_gene["gene_name"].isna().sum()
if n_missing_symbols > 0:
    print(f"⚠️  {n_missing_symbols} gene_ids did not map to any gene_name; keeping Ensembl ID for those rows.")

# Create a Series of original indices so fillna can align correctly
index_series = pd.Series(expr_gene.index, index=expr_gene.index)
expr_gene["final_name"] = expr_gene["gene_name"].fillna(index_series)

# Now set final_name as the DataFrame’s index
expr_gene = expr_gene.set_index("final_name", drop=True)

# ──────────────────────────────────────────────────────────────────────────────
# 10) Collapse duplicates on gene_name (if two gene_ids mapped to same symbol)
# ──────────────────────────────────────────────────────────────────────────────
expr_gene = (
    expr_gene
      .drop(columns=["gene_name"])   # no longer needed
      .groupby(level=0)              # collapse on gene symbol
      .sum()
)

print(f"After renaming → gene symbols, expr_gene shape = {expr_gene.shape}")
print("Example of first 5 row indices (gene symbols):", expr_gene.index[:5].tolist())

# ──────────────────────────────────────────────────────────────────────────────
# 11) Save the gene‐level TPM matrix with gene names as index
# ──────────────────────────────────────────────────────────────────────────────
expr_gene.to_parquet(GENE_PARQUET, compression="snappy")
print("✅  Gene‐level TPM (with gene_name index) written to:", GENE_PARQUET)


Protein Coding Gene level expresion matrix

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell: Filter gene‐level TPM to protein‐coding genes only
# ──────────────────────────────────────────────────────────────────────────────

from pathlib import Path
import pandas as pd
import re

# ──────────────────────────────────────────────────────────────────────────────
# Paths (adjust if needed)
# ──────────────────────────────────────────────────────────────────────────────
BASE           = Path("/content/drive/MyDrive/dmd_dmd_graph")
DATA_DIR       = BASE / "data"
PROC_DIR       = BASE / "proc"

GTF_PATH       = DATA_DIR / "gencode.v39.annotation.gtf"
GENE_PARQUET   = PROC_DIR / "expression_gene_tpm_named.parquet"
FILTERED_PQ    = PROC_DIR / "expression_gene_tpm_protein_coding.parquet"

# ──────────────────────────────────────────────────────────────────────────────
# 1) Load the gene‐level TPM matrix (indexed by gene symbol)
# ──────────────────────────────────────────────────────────────────────────────
expr_gene = pd.read_parquet(GENE_PARQUET)
print(f"Loaded expression matrix: {expr_gene.shape} (genes × samples)\n")

# ──────────────────────────────────────────────────────────────────────────────
# 2) Parse GTF to collect all protein‐coding gene names
# ──────────────────────────────────────────────────────────────────────────────
protein_coding_genes = set()
with open(GTF_PATH, "r") as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if cols[2] != "gene":
            continue
        attrs = cols[8]
        # Extract gene_type and gene_name
        type_match = re.search(r'gene_type "([^"]+)"', attrs)
        name_match = re.search(r'gene_name "([^"]+)"', attrs)
        if type_match and name_match and type_match.group(1) == "protein_coding":
            gene_name = name_match.group(1)
            protein_coding_genes.add(gene_name)

print(f"Found {len(protein_coding_genes)} protein‐coding gene symbols in GTF.\n")

# ──────────────────────────────────────────────────────────────────────────────
# 3) Filter expression matrix to only those protein‐coding gene symbols
# ──────────────────────────────────────────────────────────────────────────────
# Since expr_gene is indexed by gene symbol, we simply intersect
filtered_expr = expr_gene.loc[expr_gene.index.intersection(protein_coding_genes)].copy()
print(f"After filtering: {filtered_expr.shape} (protein‐coding genes × samples)\n")

# ──────────────────────────────────────────────────────────────────────────────
# 4) Save filtered matrix as Parquet
# ──────────────────────────────────────────────────────────────────────────────
filtered_expr.to_parquet(FILTERED_PQ, compression="snappy")
print("✅ Protein‐coding subset written to:", FILTERED_PQ)


Protein embeddings (sequence → vector)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# 1) Install required packages: `fair-esm` for ESM models, `biopython` for FASTA
#    parsing, and `requests` if not already installed.
# ────────────────────────────────────────────────────────────────────────────────

!pip install fair-esm biopython requests tqdm --quiet

# Verify installation versions
import esm, Bio, requests, tqdm
print("ESM version:", esm.__version__)
print("Biopython version:", Bio.__version__)
print("Requests version:", requests.__version__)
print("tqdm version:", tqdm.__version__)


In [ ]:
# Next cell: map gene symbols in expression_gene_tpm_protein_coding.parquet back to Ensembl IDs

import pandas as pd
import re
from pathlib import Path

# 1) Paths
BASE      = Path("/content/drive/MyDrive/dmd_dmd_graph")
PROC_DIR  = BASE / "proc"
GTF_PATH  = BASE / "data" / "gencode.v39.annotation.gtf"
PC_FP     = PROC_DIR / "expression_gene_tpm_protein_coding.parquet"

# 2) Load the protein‐coding gene‐level TPM (indexed by gene symbol)
expr_pc = pd.read_parquet(PC_FP)
print("Loaded protein‐coding TPM matrix:", expr_pc.shape)
print("Example gene symbols:", expr_pc.index[:5].tolist())

# 3) Parse GTF to rebuild gene_id → gene_name map (as in the collapse cell)
gene2name = {}
with open(GTF_PATH, "r") as gtf_handle:
    for line in gtf_handle:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if cols[2] != "gene":
            continue
        attrs = cols[8]
        m_id   = re.search(r'gene_id "([^"]+)"', attrs)
        m_name = re.search(r'gene_name "([^"]+)"', attrs)
        if m_id and m_name:
            gid   = m_id.group(1)
            gname = m_name.group(1)
            gene2name[gid] = gname

# 4) Invert to gene_name → list of Ensembl IDs
name2ensembl = {}
for ensg, gsym in gene2name.items():
    name2ensembl.setdefault(gsym, []).append(ensg)

# 5) For each gene symbol in expr_pc.index, pick the corresponding Ensembl ID(s)
mapped = []
unmapped_symbols = []

for gsym in expr_pc.index.astype(str):
    if gsym in name2ensembl:
        # If multiple Ensembl IDs share this symbol, pick the first one
        mapped.append(name2ensembl[gsym][0])
    else:
        mapped.append(None)
        unmapped_symbols.append(gsym)

print(f"→ Mapped {len(expr_pc) - len(unmapped_symbols)} symbols to Ensembl IDs")
print(f"→ {len(unmapped_symbols)} symbols could not be mapped, examples:", unmapped_symbols[:5])

# 6) Create a DataFrame of symbol → Ensembl ID
df_map_symbols = pd.DataFrame({
    "gene_symbol": expr_pc.index.astype(str),
    "ensembl_gene_id": mapped
})
print("\nExample mapping:")
print(df_map_symbols.head())

# 7) (Optional) Filter out any symbols that failed to map
df_map_valid = df_map_symbols.dropna(subset=["ensembl_gene_id"])
genes = df_map_valid["ensembl_gene_id"].tolist()
print(f"\nTotal Ensembl IDs for UniProt mapping: {len(genes)}")

# Now `genes` is a list of Ensembl Gene IDs corresponding to your protein‐coding gene symbols.


In [ ]:
import requests
import time
import re
from pathlib import Path
import pandas as pd
from io import StringIO

# ──────────────────────────────────────────────────────────────────────────────
# Step 1: Parse GTF to build ENSG → {ENSP, ENSP, …} from protein‐coding genes
# ──────────────────────────────────────────────────────────────────────────────

BASE     = Path("/content/drive/MyDrive/dmd_dmd_graph")
DATA_DIR = BASE / "data"
GTF_PATH = DATA_DIR / "gencode.v39.annotation.gtf"

gene2proteins = {}
with open(GTF_PATH, "r") as gtf_handle:
    for line in gtf_handle:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if cols[2] != "CDS":
            continue
        attrs = cols[8]
        m_gene = re.search(r'gene_id "([^"]+)"', attrs)
        m_prot = re.search(r'protein_id "([^"]+)"', attrs)
        if not (m_gene and m_prot):
            continue
        ensg = m_gene.group(1).split(".")[0]
        ensp = m_prot.group(1).split(".")[0]
        gene2proteins.setdefault(ensg, set()).add(ensp)

# ──────────────────────────────────────────────────────────────────────────────
# Step 2: Build full list of ENSP IDs for your protein‐coding ENSG list
# ──────────────────────────────────────────────────────────────────────────────

# `genes` is assumed defined earlier as the list of ~19 922 protein‐coding ENSG IDs (no version).
# If not, re-derive `genes` from expression_final.parquet as shown previously.

all_proteins = []
for g in genes:
    prot_set = gene2proteins.get(g)
    if prot_set:
        all_proteins.extend(prot_set)

all_proteins = sorted(set(all_proteins))
print(f"→ Found {len(all_proteins)} unique Ensembl Protein IDs for mapping")

# ──────────────────────────────────────────────────────────────────────────────
# Step 3: Map ENSP → UniProtKB via UniProt ID‐mapping with pagination
# ──────────────────────────────────────────────────────────────────────────────

MAP_DIR = BASE / "data" / "uniprot_mapping"
MAP_DIR.mkdir(parents=True, exist_ok=True)

submit_url = "https://rest.uniprot.org/idmapping/run"
status_url = "https://rest.uniprot.org/idmapping/status/"
result_url = "https://rest.uniprot.org/idmapping/results/"

batch_size = 500
protein_map_pairs = []

for i in range(0, len(all_proteins), batch_size):
    chunk = all_proteins[i : i + batch_size]
    print(f"→ Submitting batch {i//batch_size + 1} with {len(chunk)} ENSP IDs …")
    params = {
        "from": "Ensembl_Protein",
        "to":   "UniProtKB",
        "ids":  ",".join(chunk)
    }
    r = requests.post(submit_url, data=params)
    r.raise_for_status()
    job_id = r.json()["jobId"]
    print("   • Job ID:", job_id)

    # Poll until job is complete
    while True:
        status = requests.get(status_url + job_id).json()
        if status.get("jobStatus") == "RUNNING":
            time.sleep(3)
        else:
            break

    # Pagination: get first page (size=500)
    params_tsv = {"format": "tsv", "size": 500}
    first_page_text = requests.get(result_url + job_id, params=params_tsv).text
    lines = first_page_text.splitlines()
    all_lines = lines.copy()

    # Check for cursor
    params_json = {"format": "json", "size": 0}
    json_status = requests.get(result_url + job_id, params=params_json).json()
    cursor = json_status.get("next", "").split("cursor=")[-1] if "next" in json_status else None

    # Loop through additional pages
    while cursor:
        page_text = requests.get(
            result_url + job_id,
            params={"format": "tsv", "cursor": cursor, "size": 500}
        ).text
        page_lines = page_text.splitlines()
        all_lines.extend(page_lines[1:])  # skip repeated header
        json_status = requests.get(
            result_url + job_id,
            params={"format": "json", "cursor": cursor, "size": 0}
        ).json()
        cursor = json_status.get("next", "").split("cursor=")[-1] if "next" in json_status else None

    res = "\n".join(all_lines)

    # Save this batch’s concatenated TSV
    batch_fp = MAP_DIR / f"idmap_batch_protein_{i//batch_size + 1}.tsv"
    with open(batch_fp, "w") as fh:
        fh.write(res)
    print(f"   ✓ Saved mapping to {batch_fp}")

    # Parse TSV, take first two columns as (ENSP, UniProt)
    df_map = pd.read_csv(StringIO(res), sep="\t")
    if df_map.shape[1] < 2:
        print(f"⚠ Batch {i//batch_size + 1} returned < 2 columns; skipping.")
        continue

    df_pair = df_map.iloc[:, :2].copy()
    df_pair.columns = ["ensembl_protein_id", "uniprot_id"]
    protein_map_pairs.append(df_pair)

# Concatenate all protein→UniProt pairs
if protein_map_pairs:
    protein2uniprot = pd.concat(protein_map_pairs, ignore_index=True).drop_duplicates()
else:
    protein2uniprot = pd.DataFrame(columns=["ensembl_protein_id", "uniprot_id"])

print(f"\n→ Total protein→UniProt mappings: {protein2uniprot.shape[0]}")

# Save protein-level mapping
protein_map_fp = MAP_DIR / "ensembl_protein_to_uniprot.tsv"
protein2uniprot.to_csv(protein_map_fp, sep="\t", index=False)
print("✅ Saved protein→UniProt mapping:", protein_map_fp)

# ──────────────────────────────────────────────────────────────────────────────
# Step 4: Convert ENSP→UniProt to ENSG→UniProt by inverting gene2proteins
# ──────────────────────────────────────────────────────────────────────────────

# Build ENSP → ENSG lookup
ensp2gene = {}
for gene, prot_set in gene2proteins.items():
    for p in prot_set:
        ensp2gene[p] = gene

# Expand to gene→UniProt
records = []
for _, row in protein2uniprot.iterrows():
    ensp = row["ensembl_protein_id"]
    up   = row["uniprot_id"]
    gene = ensp2gene.get(ensp)
    if gene:
        records.append((gene, up))

df_gene2up = pd.DataFrame(records, columns=["ensembl_gene_id", "uniprot_id"]).drop_duplicates()

print("Unique ENSG IDs mapped at least once:", df_gene2up["ensembl_gene_id"].nunique())
print("Total ENSG→UniProt pairs:", df_gene2up.shape[0])

# Save final gene‐level mapping
gene_map_fp = MAP_DIR / "ensembl_gene_to_uniprot.tsv"
df_gene2up.to_csv(gene_map_fp, sep="\t", index=False)
print("✅ Saved final ENSG→UniProt mapping:", gene_map_fp)


In [ ]:
gene_map_fp = BASE / "data" / "uniprot_mapping" / "ensembl_gene_to_uniprot.tsv"
mapping_df = pd.read_csv(gene_map_fp, sep="\t")

print("Total rows in mapping_df:", mapping_df.shape[0])
print("Unique ENSG IDs mapped: ", mapping_df["ensembl_gene_id"].nunique())
print("Unique UniProt IDs   : ", mapping_df["uniprot_id"].nunique())


In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Step 5h (corrected): Download FASTA in batches of 1 000 UniProt IDs, using "OR"‐joined accession queries
# ────────────────────────────────────────────────────────────────────────────────

import requests
from pathlib import Path
from Bio import SeqIO
import pandas as pd
import re
from io import StringIO
import urllib.parse

# 5a) Paths (assume previous steps have defined BASE, DATA_DIR, PROC_DIR, etc.)
BASE      = Path("/content/drive/MyDrive/dmd_dmd_graph")
DATA_DIR  = BASE / "data"
PROC_DIR  = BASE / "proc"
GTF_PATH  = DATA_DIR / "gencode.v39.annotation.gtf"
FASTA_DIR = DATA_DIR / "uniprot_fasta"
FASTA_DIR.mkdir(parents=True, exist_ok=True)

# 5b) Load protein-coding gene‐level TPM (indexed by gene symbol)
expr_pc_fp   = PROC_DIR / "expression_gene_tpm_protein_coding.parquet"
df_pc        = pd.read_parquet(expr_pc_fp)
gene_symbols = df_pc.index.astype(str).tolist()
print("Loaded protein-coding matrix with gene symbols:", len(gene_symbols), "genes")

# 5c) Parse GTF to build mappings:
#     – gene_id → gene_name  (gene2name)
#     – gene_name → list of ENSG  (name2ensg)
#     – gene_id → set of ENSP   (gene2proteins)
gene2name     = {}
gene2proteins = {}
with open(GTF_PATH, "r") as gtf_handle:
    for line in gtf_handle:
        if line.startswith("#"):
            continue
        cols    = line.rstrip("\n").split("\t")
        feature = cols[2]
        if feature == "gene":
            attrs  = cols[8]
            m_id   = re.search(r'gene_id "([^"]+)"', attrs)
            m_name = re.search(r'gene_name "([^"]+)"', attrs)
            if m_id and m_name:
                ensg = m_id.group(1).split(".")[0]
                gsym = m_name.group(1)
                gene2name[ensg] = gsym
        elif feature == "CDS":
            attrs  = cols[8]
            m_gene = re.search(r'gene_id "([^"]+)"', attrs)
            m_prot = re.search(r'protein_id "([^"]+)"', attrs)
            if m_gene and m_prot:
                ensg = m_gene.group(1).split(".")[0]
                ensp = m_prot.group(1).split(".")[0]
                gene2proteins.setdefault(ensg, set()).add(ensp)

# Invert gene2name to gene_name → list of ENSG
name2ensg = {}
for ensg, gsym in gene2name.items():
    name2ensg.setdefault(gsym, []).append(ensg)

# 5d) Map gene symbols → ENSG
expr_ensg = []
for gsym in gene_symbols:
    expr_ensg.extend(name2ensg.get(gsym, []))
expr_ensg = sorted(set(expr_ensg))
print("Mapped gene symbols → ENSG IDs:", len(expr_ensg), "genes")

# 5e) Collect ENSP IDs for those ENSGs
expr_ensp = []
for ensg in expr_ensg:
    expr_ensp.extend(gene2proteins.get(ensg, []))
expr_ensp = sorted(set(expr_ensp))
print("Derived ENSP IDs for those genes:", len(expr_ensp), "proteins")

# 5f) Load ENSG→UniProt mapping
gene_map_fp = BASE / "data" / "uniprot_mapping" / "ensembl_gene_to_uniprot.tsv"
gene2up_df  = pd.read_csv(gene_map_fp, sep="\t")

# Filter to only those ENSGs in our expression set
gene2up_df  = gene2up_df[gene2up_df["ensembl_gene_id"].isin(expr_ensg)]

# 5g) Extract unique UniProt IDs
uniprot_ids = sorted(gene2up_df["uniprot_id"].unique())
print(f"Total unique UniProt IDs to download: {len(uniprot_ids)}")
print("Example UniProt IDs:", uniprot_ids[:5])

# 5h) Download FASTA in batches of 1 000 UniProt IDs using OR‐joined query
batch_size = 1000

for i in range(0, len(uniprot_ids), batch_size):
    batch      = uniprot_ids[i : i + batch_size]
    batch_idx  = (i // batch_size) + 1
    print(f"→ Downloading batch {batch_idx} ({len(batch)} IDs)…")

    # Build an "OR"‐joined accession query string, then URL‐encode it
    or_query   = " OR ".join(f"accession:{uid}" for uid in batch)
    encoded_q  = urllib.parse.quote(or_query)
    url        = f"https://rest.uniprot.org/uniprotkb/stream?format=fasta&query={encoded_q}"

    r = requests.get(url)
    if r.status_code != 200:
        print(f"⚠️  Failed to fetch batch {batch_idx} (HTTP {r.status_code}); skipping.")
        continue

    # Parse the multi‐FASTA and write each record as its own file
    fasta_io = StringIO(r.text)
    for rec in SeqIO.parse(fasta_io, "fasta"):
        # rec.id might come as "sp|P12345|GENENAME"; split to get P12345
        uid     = rec.id.split("|")[-1] if "|" in rec.id else rec.id
        fasta_fp = FASTA_DIR / f"{uid}.fasta"
        if fasta_fp.exists():
            continue
        with open(fasta_fp, "w") as fh:
            SeqIO.write(rec, fh, "fasta")

    print(f"   ✓ Batch {batch_idx} saved to individual FASTA files")

print("\n✅ All requested FASTA files are in", FASTA_DIR)


In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Step 5h (corrected): Download FASTA in batches of 1 000 UniProt IDs, using "OR"‐joined accession queries
# ────────────────────────────────────────────────────────────────────────────────

import requests
from pathlib import Path
from Bio import SeqIO
import pandas as pd
import re
from io import StringIO
import urllib.parse

# 5a) Paths (assume previous steps have defined BASE, DATA_DIR, PROC_DIR, etc.)
BASE      = Path("/content/drive/MyDrive/dmd_dmd_graph")
DATA_DIR  = BASE / "data"
PROC_DIR  = BASE / "proc"
GTF_PATH  = DATA_DIR / "gencode.v39.annotation.gtf"
FASTA_DIR = DATA_DIR / "uniprot_fasta"
FASTA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
%set_env TOKENIZERS_PARALLELISM=false
!pip install esm
import numpy as np
import torch

!pip install py3Dmol
import py3Dmol
from esm.sdk import client
from esm.sdk.api import ESMProtein, GenerationConfig
from esm.utils.structure.protein_chain import ProteinChain

In [ ]:
!pip install pyfastx

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Step 5b (optimized per‐file with SeqIO): Preload sequences by reading each FASTA,
# logging progress every 1000 files
# ────────────────────────────────────────────────────────────────────────────────

import time
from pathlib import Path
from Bio import SeqIO

BASE      = Path("/content/drive/MyDrive/dmd_dmd_graph")
FASTA_DIR = BASE / "data" / "uniprot_fasta"

# Gather all FASTA file paths
fasta_files = sorted(FASTA_DIR.glob("*.fasta"))
num_files   = len(fasta_files)
print(f"Preloading {num_files} individual FASTA files with SeqIO…")

start = time.time()
all_sequences = {}

for idx, fp in enumerate(fasta_files, start=1):
    uid = fp.stem
    rec = next(SeqIO.parse(str(fp), "fasta"))
    all_sequences[uid] = str(rec.seq)

    if idx % 1000 == 0 or idx == num_files:
        elapsed = time.time() - start
        print(f"   • Loaded {idx}/{num_files} sequences in {elapsed:.1f} s")

print(f"\n   ✓ Finished preloading {num_files} sequences in {time.time() - start:.1f} s.\n")

# 'all_sequences' now maps each UniProt ID → its sequence string.


In [ ]:
import pickle

with open(PROC_DIR / "all_sequences.pkl", "rb") as f:
    all_sequences = pickle.load(f)


In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Step 5d–5g: Embed all sequences using the preloaded 'all_sequences' dict
# (replacing the old SeqIO.index usage)
# ────────────────────────────────────────────────────────────────────────────────

import time
import torch
import numpy as np
from pathlib import Path
import esm

# 5a) Paths
BASE     = Path("/content/drive/MyDrive/dmd_dmd_graph")
OUT_DIR  = BASE / "proc"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 5b) We assume 'all_sequences' was already loaded (either via pyfastx or fallback).
# If not in memory, load from pickle as shown previously:
# import pickle
# with open(OUT_DIR / "all_sequences.pkl", "rb") as f:
#     all_sequences = pickle.load(f)

# For this example, let's assume 'all_sequences' is already defined:
# all_sequences = { "P01234": "MTEITAAMVK...", ... }

uids        = sorted(all_sequences.keys())
num_files   = len(uids)
print(f"Embedding {num_files} sequences…\n")

# 5c) Confirm GPU availability
assert torch.cuda.is_available(), "CUDA not available. Please select a GPU runtime."
device = torch.device("cuda")
print(f"Using device: {device} – {torch.cuda.get_device_name(0)}\n")

# 5d) Load pretrained ESM-2 (150M) model onto GPU
model, alphabet = esm.pretrained.load_model_and_alphabet("esm2_t30_150M_UR50D")
model = model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

# 5e) Embed in batches of 64 sequences, logging progress
batch_size    = 64
MAX_LEN       = 1024  # truncate any sequence longer than this
total_batches = (num_files + batch_size - 1) // batch_size

embeddings    = {}
overall_start = time.time()

for batch_idx in range(total_batches):
    i_start    = batch_idx * batch_size
    i_end      = min(i_start + batch_size, num_files)
    batch_uids = uids[i_start : i_end]
    batch_data = []

    # Log start of this batch
    print(f"→ Batch {batch_idx+1}/{total_batches} (records {i_start+1}–{i_end}), preparing…")
    batch_start = time.time()

    # Build (uid, seq) list for this batch using 'all_sequences'
    for uid in batch_uids:
        seq = all_sequences[uid]
        if len(seq) > MAX_LEN:
            seq = seq[:MAX_LEN]
        batch_data.append((uid, seq))

    # Convert to tokens and move to GPU
    labels, seqs, tokens = batch_converter(batch_data)
    tokens = tokens.to(device, non_blocking=True)

    # Model forward pass
    with torch.no_grad():
        results   = model(tokens, repr_layers=[model.num_layers], return_contacts=False)
    token_reps = results["representations"][model.num_layers]

    # Extract and store embeddings
    for b, (uid, seq) in enumerate(batch_data):
        rep = token_reps[b, 1 : len(seq) + 1]           # drop BOS/EOS tokens
        vec = rep.mean(0).cpu().numpy()                 # shape: (640,)
        embeddings[uid] = vec

    # Clear GPU cache, log batch time
    torch.cuda.empty_cache()
    batch_elapsed = time.time() - batch_start
    print(f"   ✓ Completed batch {batch_idx+1}/{total_batches} in {batch_elapsed:.1f} s\n")

total_elapsed = time.time() - overall_start
print(f"✅ All {num_files} sequences embedded in {total_elapsed/60:.1f} min.\n")

# 5f) Stack all embeddings into an (N × 640) array and save to PROC directory
all_ids     = sorted(embeddings.keys())
all_vectors = np.stack([embeddings[uid] for uid in all_ids], axis=0)
out_fp      = OUT_DIR / "esm2_150M_proj.npy"
np.save(out_fp, all_vectors)

print(f"Saved ESM-2 (150 M) embeddings for {len(all_ids)} proteins → {out_fp}")
print("Embedding array shape:", all_vectors.shape)
print("Example protein IDs:", all_ids[:10], "…")


Protein Edges

In [ ]:
import requests
from pathlib import Path
import pandas as pd
from requests.exceptions import ConnectionError, RequestException
import glob

# ────────────────────────────────────────────────────────────────────────────────
# Utility: POST with retries on ConnectionError or any RequestException (no waits)
# ────────────────────────────────────────────────────────────────────────────────
def post_with_retries(url, data, timeout=60, max_retries=3):
    attempt = 0
    while attempt < max_retries:
        try:
            resp = requests.post(url, data=data, timeout=timeout)
            resp.raise_for_status()
            return resp
        except ConnectionError:
            attempt += 1
            if attempt >= max_retries:
                print(f"⚠️ Connection to {url} failed after {max_retries} attempts. Skipping.")
                return None
        except RequestException as e:
            # If it's an HTTP error (status ≥400), return resp so caller can check status_code
            if hasattr(e, "response") and e.response is not None:
                return e.response
            attempt += 1
            if attempt >= max_retries:
                print(f"⚠️ Request to {url} failed ({e.__class__.__name__}) after {max_retries} attempts. Skipping.")
                return None
    return None

# ────────────────────────────────────────────────────────────────────────────────
# Step: For each UniProt ID, map → STRING ID, then immediately fetch edges (one at a time).
# After every 1000 UniProt IDs, dump collected edges to a partial TSV and clear buffer.
# Finally, combine all partial TSVs into one and delete partials.
# ────────────────────────────────────────────────────────────────────────────────

mapping_fp  = Path("/content/drive/MyDrive/dmd_dmd_graph/data/uniprot_mapping/ensembl_gene_to_uniprot.tsv")
gene2up_df  = pd.read_csv(mapping_fp, sep="\t")
all_uniprot = sorted(gene2up_df["uniprot_id"].dropna().unique())
print(f"Loaded {len(all_uniprot)} unique UniProt IDs.\n")

STRING_BASE = "https://string-db.org/api/tsv"
GET_IDS_URL = f"{STRING_BASE}/get_string_ids"
NETWORK_URL = f"{STRING_BASE}/network"
SPECIES     = 9606
REQ_SCORE   = 400

# Create a folder for partial outputs
partial_dir = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/ppi_partials")
partial_dir.mkdir(parents=True, exist_ok=True)

string_mappings = {}
all_edges_buffer = []
batch_size = 1000
part_idx = 1

for idx, uni in enumerate(all_uniprot, start=1):
    print(f"[{idx}/{len(all_uniprot)}] Mapping UniProt '{uni}' → STRING ID")
    # 1) Map this single UniProt to a STRING ID
    data_map = {
        "identifiers": uni,
        "species":      SPECIES,
        "limit":        1,
        "echo_query":   0
    }
    resp_map = post_with_retries(GET_IDS_URL, data_map, timeout=60, max_retries=3)
    if resp_map is None:
        print(f"    → Network error. Skipping '{uni}'.\n")
        continue

    if resp_map.status_code >= 400:
        print(f"    → HTTP {resp_map.status_code} returned. No mapping for '{uni}'. Skipping.\n")
        continue

    lines_map = resp_map.text.strip().splitlines()
    if len(lines_map) <= 1:
        print(f"    → No match found in STRING. Skipping '{uni}'.\n")
        continue

    cols_map = lines_map[1].split("\t")
    if len(cols_map) < 2:
        print(f"    → Unexpected response format. Skipping '{uni}'.\n")
        continue

    sid = cols_map[1]  # column 1 is the STRING ID
    string_mappings[uni] = sid
    print(f"    → Mapped '{uni}' → '{sid}'.\n")

    # 2) Immediately fetch PPI edges for this STRING ID
    print(f"    [→] Querying /network for STRING ID '{sid}'")
    data_net = {
        "identifiers":    sid,
        "species":        SPECIES,
        "required_score": REQ_SCORE
    }
    resp_net = post_with_retries(NETWORK_URL, data_net, timeout=60, max_retries=3)
    if resp_net is None:
        print(f"        → Network error fetching edges for '{sid}'. Skipping.\n")
        continue

    if resp_net.status_code >= 400:
        print(f"        → HTTP {resp_net.status_code} returned. No edges for '{sid}'. Skipping.\n")
        continue

    lines_net = resp_net.text.strip().splitlines()
    if len(lines_net) <= 1:
        print(f"        → No edges found for '{sid}'.\n")
        continue

    count = 0
    for line in lines_net[1:]:
        parts = line.split("\t")
        if len(parts) < 3:
            continue
        protA_full, protB_full, score_str = parts[0], parts[1], parts[-1]
        if protA_full.startswith(f"{SPECIES}."):
            protA_full = protA_full.split(".", 1)[1]
        if protB_full.startswith(f"{SPECIES}."):
            protB_full = protB_full.split(".", 1)[1]
        try:
            score_val = float(score_str)
        except ValueError:
            continue
        all_edges_buffer.append((protA_full, protB_full, score_val))
        count += 1

    print(f"        → Retrieved {count} edges for '{sid}'.\n")

    # Every batch_size UniProt IDs, write out a partial TSV
    if idx % batch_size == 0 or idx == len(all_uniprot):
        if all_edges_buffer:
            partial_fp = partial_dir / f"ppi_edges_part_{part_idx}.tsv"
            print(f"    ➜ Writing {len(all_edges_buffer)} edges to partial file {partial_fp.name}")
            part_df = (
                pd.DataFrame(all_edges_buffer, columns=["stringA","stringB","combined_score"])
                  .drop_duplicates(subset=["stringA","stringB"])
            )
            part_df.to_csv(partial_fp, sep="\t", index=False)
            all_edges_buffer.clear()
            part_idx += 1

# ────────────────────────────────────────────────────────────────────────────────
# Combine all partial TSVs into one final file, filter to keep only edges
# where both endpoints were queried, then reverse-map and save final TSV.
# ────────────────────────────────────────────────────────────────────────────────

# Read and concatenate all partial files
partial_files = sorted(glob.glob(str(partial_dir / "ppi_edges_part_*.tsv")))
if not partial_files:
    print("❗️ No partial files found; nothing to combine.")
else:
    dfs = []
    for pf in partial_files:
        dfs.append(pd.read_csv(pf, sep="\t"))
    combined_df = pd.concat(dfs, ignore_index=True)

    # Deduplicate across all partials
    combined_df = combined_df.drop_duplicates(subset=["stringA","stringB"]).reset_index(drop=True)

    # Build reverse map from STRING ID → UniProt
    rev_map = {sid: uni for uni, sid in string_mappings.items()}

    # Keep only edges where both endpoints were in rev_map
    mask = combined_df["stringA"].isin(rev_map) & combined_df["stringB"].isin(rev_map)
    combined_df = combined_df.loc[mask].copy()

    # Reverse-map
    combined_df["protein1"] = combined_df["stringA"].map(rev_map)
    combined_df["protein2"] = combined_df["stringB"].map(rev_map)

    final_df = combined_df[["protein1","protein2","combined_score"]]

    # Write final aggregated TSV
    out_dir = Path("/content/drive/MyDrive/dmd_dmd_graph/proc")
    out_dir.mkdir(parents=True, exist_ok=True)
    final_fp = out_dir / "ppi_edges.tsv"
    final_df.to_csv(final_fp, sep="\t", index=False)
    print(f"\n✅ Wrote {final_df.shape[0]} unique PPI edges → {final_fp}")

    # Delete partial files
    for pf in partial_files:
        Path(pf).unlink()
    print("✅ Deleted intermediate partial files.")


In [ ]:
import glob
import pandas as pd
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────────────
# 1) Load Ensembl‐gene → UniProt mapping
# ────────────────────────────────────────────────────────────────────────────────
mapping_fp = Path(
    "/content/drive/MyDrive/dmd_dmd_graph/data/uniprot_mapping/ensembl_protein_to_uniprot.tsv"
)
# This file must have exactly two columns: ensembl_protein_idand uniprot_id
emap_df = pd.read_csv(mapping_fp, sep="\t", usecols=["ensembl_protein_id", "uniprot_id"])

# Build a dictionary: { "ENSG00000...": "P12345", ... }
rev_map = dict(zip(emap_df["ensembl_protein_id"], emap_df["uniprot_id"]))


# ────────────────────────────────────────────────────────────────────────────────
# 2) Find all partial files under ppi_partials/
# ────────────────────────────────────────────────────────────────────────────────
partial_dir = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/ppi_partials")
partial_files = sorted(glob.glob(str(partial_dir / "ppi_edges_part_*.tsv")))

if not partial_files:
    print("❗️ No partial files found under ppi_partials/. Nothing to combine.")
else:
    # ──────────────────────────────────────────────────────────────────────────
    # 3) Read & concatenate every partial
    # ──────────────────────────────────────────────────────────────────────────
    dfs = [pd.read_csv(pf, sep="\t") for pf in partial_files]
    combined_df = pd.concat(dfs, ignore_index=True)

    # ──────────────────────────────────────────────────────────────────────────
    # 4) Deduplicate across all partials (stringA & stringB are Ensembl gene IDs)
    # ──────────────────────────────────────────────────────────────────────────
    combined_df = (
        combined_df.drop_duplicates(subset=["stringA", "stringB"])
        .reset_index(drop=True)
    )

    # ──────────────────────────────────────────────────────────────────────────
    # 5) Keep only edges where both endpoints appear in rev_map (i.e. both are Ensembl IDs)
    # ──────────────────────────────────────────────────────────────────────────
    mask = combined_df["stringA"].isin(rev_map) & combined_df["stringB"].isin(rev_map)
    filtered_df = combined_df.loc[mask].copy()

    # ──────────────────────────────────────────────────────────────────────────
    # 6) Reverse‐map from Ensembl → UniProt
    # ──────────────────────────────────────────────────────────────────────────
    filtered_df["protein1"] = filtered_df["stringA"].map(rev_map)
    filtered_df["protein2"] = filtered_df["stringB"].map(rev_map)

    final_df = filtered_df[["protein1", "protein2", "combined_score"]]

    # ──────────────────────────────────────────────────────────────────────────
    # 7) Write the final aggregated TSV
    # ──────────────────────────────────────────────────────────────────────────
    out_dir = Path("/content/drive/MyDrive/dmd_dmd_graph/proc")
    out_dir.mkdir(parents=True, exist_ok=True)
    final_fp = out_dir / "ppi_edges.tsv"
    final_df.to_csv(final_fp, sep="\t", index=False)
    print(f"\n✅ Wrote {final_df.shape[0]} unique PPI edges → {final_fp}")

    # We do NOT delete the partial files here.
    print("✅ Partial files remain intact under ppi_partials/")


Gene Ontology annotations (multi-hot)

In [ ]:
import pandas as pd
import gzip
import requests
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────────────
# 1) Paths & URLs
# ────────────────────────────────────────────────────────────────────────────────
BASE_DIR       = Path("/content/drive/MyDrive/dmd_dmd_graph")
GO_DIR         = BASE_DIR / "data/go"
GO_DIR.mkdir(parents=True, exist_ok=True)

# 1a) List of protein‐coding gene symbols (from your filtered TPM matrix)
#     Assumes you’ve already run the code to produce:
#         /content/drive/MyDrive/dmd_dmd_graph/proc/expression_gene_tpm_protein_coding.parquet
expr_pc_fp     = BASE_DIR / "proc/expression_gene_tpm_protein_coding.parquet"

# 1b) GO Consortium GAF for human (download to GO_DIR)
GO_HUMAN_GAF   = GO_DIR / "goa_human.gaf.gz"
GAF_URL        = "http://current.geneontology.org/annotations/goa_human.gaf.gz"

# 1c) Output parquet
OUT_FP         = BASE_DIR / "proc/go_matrix.parquet"

# ────────────────────────────────────────────────────────────────────────────────
# 2) Download human GAF if not already present
# ────────────────────────────────────────────────────────────────────────────────
if not GO_HUMAN_GAF.exists():
    print("Downloading goa_human.gaf.gz …")
    with requests.get(GAF_URL, stream=True) as r:
        r.raise_for_status()
        with open(GO_HUMAN_GAF, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print("Downloaded.")

# ────────────────────────────────────────────────────────────────────────────────
# 3) Load list of protein‐coding genes (symbols)
# ────────────────────────────────────────────────────────────────────────────────
expr_pc = pd.read_parquet(expr_pc_fp)
# The index is gene symbols
protein_coding_genes = set(expr_pc.index.tolist())
print(f"Found {len(protein_coding_genes)} protein‐coding genes in TPM matrix.")

# ────────────────────────────────────────────────────────────────────────────────
# 4) Parse GAF, collect (gene_symbol → GO_ID) pairs
# ────────────────────────────────────────────────────────────────────────────────
gene2go = []  # list of tuples (gene_symbol, go_id)
with gzip.open(GO_HUMAN_GAF, "rt") as f:
    for line in f:
        if line.startswith("!"):
            continue
        cols = line.strip().split("\t")
        # GAF columns (0-based):
        #   2: DB_Object_Symbol (gene symbol)
        #   4: Qualifier (ignore if contains “NOT”)
        #   5: GO_ID
        #   8: Aspect (P/B/F)—optional, but we can include all aspects
        gene_sym = cols[2]
        qualifier = cols[4]
        go_id     = cols[4+1]  # actually column 4 is qualifier; column 5 is GO_ID
        go_id     = cols[5]
        # Skip negated annotations
        if "NOT" in qualifier:
            continue
        # Only keep protein‐coding genes we care about
        if gene_sym in protein_coding_genes:
            gene2go.append((gene_sym, go_id))

print(f"Collected {len(gene2go)} raw (gene, GO) pairs.")

# ────────────────────────────────────────────────────────────────────────────────
# 5) Build a DataFrame and pivot to multi‐hot
# ────────────────────────────────────────────────────────────────────────────────
df = pd.DataFrame(gene2go, columns=["gene_symbol", "go_id"])
df["value"] = 1

# Pivot: index=gene_symbol, columns=go_id → fill missing with 0
go_matrix = df.pivot_table(
    index="gene_symbol",
    columns="go_id",
    values="value",
    aggfunc="max",
    fill_value=0
)

print(f"GO matrix shape: {go_matrix.shape} (genes × GO terms)")

# ────────────────────────────────────────────────────────────────────────────────
# 6) Save to Parquet
# ────────────────────────────────────────────────────────────────────────────────
go_matrix.to_parquet(OUT_FP, compression="snappy")
print(f"✅ go_matrix.parquet written to {OUT_FP}")


Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import re
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────────────
# 1) Load expression & GO to define “target_genes”
# ────────────────────────────────────────────────────────────────────────────────
expr_fp    = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/expression_gene_tpm_protein_coding.parquet")
expr_df    = pd.read_parquet(expr_fp)
genes_expr = set(expr_df.index)  # ~19 922 gene symbols

go_fp    = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/go_matrix.parquet")
go_df    = pd.read_parquet(go_fp)
genes_go = set(go_df.index)      # ~18 520 gene symbols

target_genes = sorted(genes_expr & genes_go)
print("Target gene set size:", len(target_genes))  # ≈ 18 520


# ────────────────────────────────────────────────────────────────────────────────
# 2) Load raw ESM embeddings (75 482 × 640)
# ────────────────────────────────────────────────────────────────────────────────
esm_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/esm2_150M_proj.npy")
X_raw  = np.load(esm_fp)
print("ESM raw shape:", X_raw.shape)  # should be (75482, 640)


# ────────────────────────────────────────────────────────────────────────────────
# 3) Build ensembl_gene_id → gene_symbol from the GTF
# ────────────────────────────────────────────────────────────────────────────────
gtf_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/data/gencode.v39.annotation.gtf")
ensembl_to_symbol = {}

with open(gtf_fp, "r") as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue
        cols = line.rstrip().split("\t")
        if cols[2] != "gene":
            continue
        attrs = cols[8]
        id_match   = re.search(r'gene_id "([^"]+)"', attrs)
        name_match = re.search(r'gene_name "([^"]+)"', attrs)
        type_match = re.search(r'gene_type "([^"]+)"', attrs)
        if id_match and name_match and type_match:
            gene_id   = id_match.group(1)    # could be "ENSG00000141510.16"
            gene_name = name_match.group(1)  # e.g. "TP53"
            gene_type = type_match.group(1)
            if gene_type == "protein_coding":
                # strip off any ".<version>" suffix:
                base_id = gene_id.split(".", 1)[0]
                ensembl_to_symbol[base_id] = gene_name

print("Number of protein-coding ENSG in GTF (versions stripped):", len(ensembl_to_symbol))
# Expect ≈ 19 964 (or whatever v39 has).


# ────────────────────────────────────────────────────────────────────────────────
# 4) Load “ensembl_gene_to_uniprot.tsv” and drop duplicates
#    (Use exactly this file path:)
#    /content/drive/MyDrive/dmd_dmd_graph/data/uniprot_mapping/ensembl_gene_to_uniprot.tsv
# ────────────────────────────────────────────────────────────────────────────────
gene_map_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/data/uniprot_mapping/ensembl_gene_to_uniprot.tsv")
gene_map_df = pd.read_csv(
    gene_map_fp,
    sep="\t",
    usecols=["ensembl_gene_id", "uniprot_id"]
)
print("gene_map_df.shape (before processing):", gene_map_df.shape)
# e.g. (75 674, 2)

# 4A) Strip version suffix from ensembl_gene_id, if present:
gene_map_df["ensembl_gene_id"] = gene_map_df["ensembl_gene_id"]\
    .astype(str)\
    .str.split(".", n=1)\
    .str[0]

# 4B) Drop duplicates so that each uniprot_id appears exactly once (keep first)
gene_map_nodup = gene_map_df.drop_duplicates(subset="uniprot_id", keep="first")
print("gene_map_nodup.shape (after dedup):", gene_map_nodup.shape)
# should now be (75 482, 2)

uniprot_list = gene_map_nodup["uniprot_id"].tolist()
print("Unique UniProt count (should be 75 482):", len(uniprot_list))

assert len(uniprot_list) == X_raw.shape[0], (
    f"Length mismatch: {len(uniprot_list)} UniProts vs {X_raw.shape[0]} ESM rows"
)
print("✅ gene_map_nodup matches ESM row‐count.")


# ────────────────────────────────────────────────────────────────────────────────
# 5) DEBUG: see how many ENSG now overlap
# ────────────────────────────────────────────────────────────────────────────────
ensg_from_map = set(gene_map_nodup["ensembl_gene_id"])
common_ensg  = ensg_from_map & set(ensembl_to_symbol.keys())
print("Number of ENSG in both gene_map_nodup and GTF:", len(common_ensg), "/", len(ensg_from_map))
# Should be nonzero (≈ 19 964).


# ────────────────────────────────────────────────────────────────────────────────
# 6) Build UniProt → gene_symbol map:
#      UniProt → (stripped ENSG) → gene_symbol
# ────────────────────────────────────────────────────────────────────────────────
up2gene = {}
for _, row in gene_map_nodup.iterrows():
    ensg   = row["ensembl_gene_id"]   # e.g. "ENSG00000141510"
    up     = row["uniprot_id"]        # e.g. "P04637"
    symbol = ensembl_to_symbol.get(ensg)
    if symbol is not None:
        up2gene[up] = symbol

print("Total UniProt → gene_symbol mappings:", len(up2gene))
# Expect ≈ 19 964 (all protein-coding genes that have UniProt IDs).


# ────────────────────────────────────────────────────────────────────────────────
# 7) Collapse isoform embeddings → gene by averaging
# ────────────────────────────────────────────────────────────────────────────────
# 7A) Build a list of length 75 482, where each entry is up2gene[uniprot_list[i]] or None
row_genes = [ up2gene.get(u, None) for u in uniprot_list ]

# 7B) Create a DataFrame of shape (75 482 × 640), indexed by gene_symbol
df_emb = pd.DataFrame(X_raw, index=row_genes)

# Drop any rows whose index is None (these UniProts didn’t map to a protein_coding gene)
df_emb = df_emb[df_emb.index.notna()]
print("After dropping unmapped proteins →",
      df_emb.shape[0], "rows remain, covering", df_emb.index.nunique(), "distinct genes")

# 7C) Group by gene_symbol and average
esm_by_gene = df_emb.groupby(level=0).mean()
print("After averaging isoforms →",
      esm_by_gene.shape[0], "genes,", esm_by_gene.shape[1], "dims")
# Should be ≈ 19 964 × 640


# ────────────────────────────────────────────────────────────────────────────────
# 8) Keep only “target_genes” (intersection of TPM & GO) and reorder
# ────────────────────────────────────────────────────────────────────────────────

# Compute which target_genes actually made it into esm_by_gene:
genes_common = esm_by_gene.index.intersection(target_genes)
missing     = set(target_genes) - set(genes_common)
print(f"Found {len(genes_common)} of {len(target_genes)} target_genes in esm_by_gene.")
if missing:
    print(f"Warning: {len(missing)} genes were missing from esm_by_gene (e.g. {list(missing)[:5]} …)")

esm_aligned = esm_by_gene.loc[genes_common].copy()
print("ESM aligned shape (genes × 640):", esm_aligned.shape)  # should be (~18520 × 640)

expr_aligned = expr_df.loc[esm_aligned.index]
go_aligned   = go_df.loc[esm_aligned.index]

print("Shapes after final alignment:")
print("  expression:", expr_aligned.shape)  # e.g. (~18520 × n_samples)
print("  GO matrix: ", go_aligned.shape)    # e.g. (~18520 × n_GO_terms)
print("  ESM embds:", esm_aligned.shape)    # (~18520 × 640)



In [ ]:
import pandas as pd
from pathlib import Path


# ────────────────────────────────────────────────────────────────────────────────
# 1b) Rename columns so they don’t clash
# ────────────────────────────────────────────────────────────────────────────────
# For example: prefix all expression columns with "TPM_", all GO columns with "GO_",
# and all ESM dims with "ESMdim_".

expr_cols_renamed = {c: f"TPM_{c}" for c in expr_aligned.columns}
expr_aligned = expr_aligned.rename(columns=expr_cols_renamed)

go_cols_renamed = {c: f"GO_{c}" for c in go_aligned.columns}
go_aligned = go_aligned.rename(columns=go_cols_renamed)

esm_cols_renamed = {c: f"ESMdim_{c}" for c in esm_aligned.columns}
esm_aligned = esm_aligned.rename(columns=esm_cols_renamed)

# ────────────────────────────────────────────────────────────────────────────────
# 1c) Concatenate horizontally (axis=1); index is gene_symbol
# ────────────────────────────────────────────────────────────────────────────────
node_table = pd.concat([expr_aligned, go_aligned, esm_aligned], axis=1)
print("Final node_table shape:", node_table.shape)
# Expect: (18479, 6 + 60687 + 640) = (18479, 61333)

# ────────────────────────────────────────────────────────────────────────────────
# 1d) Save to disk (Parquet is recommended; it keeps types and is fast to load)
# ────────────────────────────────────────────────────────────────────────────────
out_dir = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/node_data")
out_dir.mkdir(parents=True, exist_ok=True)

node_fp = out_dir / "node_features.parquet"
node_table.to_parquet(node_fp, compression="snappy")
print("✅ Wrote node_features →", node_fp)


In [ ]:
import pandas as pd
import re
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────────────
# 1) Load raw PPI edges (these are already in UniProt space)
# ────────────────────────────────────────────────────────────────────────────────
ppi_fp    = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/ppi_edges.tsv")
raw_edges = pd.read_csv(ppi_fp, sep="\t")
print("Raw PPI columns:", raw_edges.columns.tolist())
print("Example rows:\n", raw_edges.head())
print(f"Total raw edges: {len(raw_edges):,}\n")

# ────────────────────────────────────────────────────────────────────────────────
# 2) Build a UniProt → ENSG → gene_symbol dictionary
# ────────────────────────────────────────────────────────────────────────────────

# 2a) Read ensembl_gene_to_uniprot.tsv (columns: ensembl_gene_id, uniprot_id)
gene_map_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/data/uniprot_mapping/ensembl_gene_to_uniprot.tsv")
gene_map_df = pd.read_csv(
    gene_map_fp,
    sep="\t",
    usecols=["ensembl_gene_id", "uniprot_id"]
)

# Strip version suffix from ENSEMBL gene IDs (e.g. "ENSG00000141510.16" → "ENSG00000141510")
gene_map_df["ensembl_gene_id"] = (
    gene_map_df["ensembl_gene_id"].astype(str).str.split(".", n=1).str[0]
)

# Drop duplicates so each UniProt appears exactly once
gene_map_nodup = gene_map_df.drop_duplicates(subset="uniprot_id", keep="first").copy()
print("gene_map_nodup.shape:", gene_map_nodup.shape, "(unique UniProt → ENSG)")

# 2b) Load GTF and build ENSG → gene_symbol (protein_coding only)
gtf_fp          = Path("/content/drive/MyDrive/dmd_dmd_graph/data/gencode.v39.annotation.gtf")
ensembl_to_sym  = {}
with open(gtf_fp, "r") as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue
        cols = line.rstrip().split("\t")
        if cols[2] != "gene":
            continue
        attrs = cols[8]
        id_match   = re.search(r'gene_id "([^"]+)"', attrs)
        name_match = re.search(r'gene_name "([^"]+)"', attrs)
        type_match = re.search(r'gene_type "([^"]+)"', attrs)
        if id_match and name_match and type_match:
            ensg_full  = id_match.group(1)          # e.g. "ENSG00000141510.16"
            gene_type  = type_match.group(1)        # e.g. "protein_coding"
            if gene_type == "protein_coding":
                ensg_base   = ensg_full.split(".", 1)[0]
                gene_symbol = name_match.group(1)
                ensembl_to_sym[ensg_base] = gene_symbol

print("Number of protein_coding ENSGs in GTF:", len(ensembl_to_sym))

# 2c) Build UniProt → gene_symbol (only for protein_coding)
# Merge gene_map_nodup (UniProt→ENSG) with ensembl_to_sym (ENSG→symbol)
uniprot_to_symbol = {}
for _, row in gene_map_nodup.iterrows():
    uni = row["uniprot_id"]
    ensg = row["ensembl_gene_id"]        # stripped version
    sym = ensembl_to_sym.get(ensg)
    if sym:
        uniprot_to_symbol[uni] = sym

print("Total UniProt → gene_symbol entries:", len(uniprot_to_symbol))

# ────────────────────────────────────────────────────────────────────────────────
# 3) Map each raw PPI row from UniProt→gene_symbol and filter
# ────────────────────────────────────────────────────────────────────────────────
raw_edges["gene1"] = raw_edges["protein1"].map(uniprot_to_symbol)
raw_edges["gene2"] = raw_edges["protein2"].map(uniprot_to_symbol)

# Drop any edge where either endpoint failed to map
edges_mapped = raw_edges.dropna(subset=["gene1", "gene2"]).copy()
print("After dropping unmapped UniProts →", edges_mapped.shape[0], "edges remain")

# 3a) Load your final node_features to get the 18 479 genes
node_fp        = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/node_data/node_features.parquet")
node_table     = pd.read_parquet(node_fp)
genes_in_nodes = set(node_table.index)  # 18 479 gene symbols

# 3b) Keep only edges where both endpoints are in genes_in_nodes
mask_keep      = edges_mapped["gene1"].isin(genes_in_nodes) & edges_mapped["gene2"].isin(genes_in_nodes)
edges_filtered = edges_mapped.loc[mask_keep, ["gene1", "gene2", "combined_score"]].copy()
print("Edges after filtering to the 18 479 genes →", edges_filtered.shape[0], "rows")

# 3c) Canonicalize undirected edges (gene1 < gene2) and drop duplicates
edges_filtered["gA"] = edges_filtered[["gene1", "gene2"]].min(axis=1)
edges_filtered["gB"] = edges_filtered[["gene1", "gene2"]].max(axis=1)

edges_final = (
    edges_filtered
    .loc[:, ["gA", "gB", "combined_score"]]
    .drop_duplicates(subset=["gA", "gB"])
    .rename(columns={"gA": "gene1", "gB": "gene2"})
    .reset_index(drop=True)
)
print("✅ Final PPI edges (gene symbols):", edges_final.shape[0], "unique edges")

# ────────────────────────────────────────────────────────────────────────────────
# 4) Save the gene‐based PPI edge list
# ────────────────────────────────────────────────────────────────────────────────
out_dir = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/ppi_gene_edges")
out_dir.mkdir(parents=True, exist_ok=True)

edge_fp = out_dir / "ppi_edges_genes.tsv"
edges_final.to_csv(edge_fp, sep="\t", index=False)
print("✅ Wrote gene-based PPI edges →", edge_fp)


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────────────
# 1) Load data
# ────────────────────────────────────────────────────────────────────────────────
node_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/node_data/node_features.parquet")
edge_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/ppi_gene_edges/ppi_edges_genes.tsv")

# Load full node‐features DataFrame (index=gene_symbol)
node_df = pd.read_parquet(node_fp)

# Identify which columns are expression (first 6) and which are ESM (last 640)
expr_cols = node_df.columns[:6]
esm_cols  = node_df.columns[-640:]

# Build a smaller feature matrix for PCA:
#   shape = (n_genes, 6 + 640) = (~18 479 × 646)
feat_df = node_df.loc[:, expr_cols.tolist() + esm_cols.tolist()]

# Load PPI edges and build NetworkX graph
edges_df = pd.read_csv(edge_fp, sep="\t")
G = nx.Graph()
G.add_nodes_from(node_df.index)
G.add_weighted_edges_from(edges_df.values)  # (gene1, gene2, combined_score)

# Precompute degree (for coloring)
deg_series = pd.Series(dict(G.degree()), name="degree")
deg_norm   = (deg_series - deg_series.min()) / (deg_series.max() - deg_series.min())

# ────────────────────────────────────────────────────────────────────────────────
# 2) Degree Distribution Histogram
# ────────────────────────────────────────────────────────────────────────────────
plt.figure(figsize=(6,4))
plt.hist(deg_series.values, bins=50, color="skyblue", edgecolor="black")
plt.xlabel("Degree")
plt.ylabel("Number of Genes")
plt.title("Degree Distribution of DMD Interactome")
plt.tight_layout()
# plt.savefig("/content/drive/MyDrive/dmd_dmd_graph/figures/degree_histogram.png", dpi=300)
plt.show()

# ────────────────────────────────────────────────────────────────────────────────
# 3) PCA on (expression + ESM) → 2D, colored by normalized degree
# ────────────────────────────────────────────────────────────────────────────────
X = feat_df.values  # shape = (18479, 646)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
pca_df = pd.DataFrame(X_pca, index=feat_df.index, columns=["PC1","PC2"])

plt.figure(figsize=(6,6))
sc = plt.scatter(
    pca_df["PC1"],
    pca_df["PC2"],
    c=deg_norm.loc[pca_df.index],
    cmap="viridis",
    s=10,
    alpha=0.8,
    linewidths=0
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of (Expression + ESM) Features\nColored by Degree")
cbar = plt.colorbar(sc, fraction=0.046, pad=0.04)
cbar.set_label("Normalized Degree", rotation=270, labelpad=15)
plt.tight_layout()
# plt.savefig("/content/drive/MyDrive/dmd_dmd_graph/figures/pca_expression_esm.png", dpi=300)
plt.show()

# ────────────────────────────────────────────────────────────────────────────────
# 4) Subnetwork of Top 50 Hub Genes
# ────────────────────────────────────────────────────────────────────────────────
top50 = deg_series.nlargest(50).index.tolist()
subG  = G.subgraph(top50)

plt.figure(figsize=(8,8))
pos = nx.spring_layout(subG, seed=42)
nx.draw_networkx_nodes(subG, pos, node_size=100, node_color="red", alpha=0.8)
nx.draw_networkx_edges(subG, pos, alpha=0.5, edge_color="gray")
nx.draw_networkx_labels(subG, pos, font_size=8)
plt.title("Subnetwork of Top 50 Hub Genes")
plt.axis("off")
plt.tight_layout()
# plt.savefig("/content/drive/MyDrive/dmd_dmd_graph/figures/top50_subnetwork.png", dpi=300)
plt.show()


Visualization

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────────────
# 0) Define an extended set of DMD‐related biomarkers to highlight
# ────────────────────────────────────────────────────────────────────────────────
known_dmd_genes = [
    "DMD", "CKM", "SGCA", "SGCB", "SGCD",
    "CAPN3", "DES", "MYH7", "MYOG", "ACTA1",
    "LPIN1", "MYH2", "MYH1", "MYL1", "MYL3",
    "FABP3", "TNNT1", "TNNT3", "TPM1", "TPM2"
]

# ────────────────────────────────────────────────────────────────────────────────
# 1) Load final node‐features and PPI edges
# ────────────────────────────────────────────────────────────────────────────────
node_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/node_data/node_features.parquet")
edge_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/ppi_gene_edges/ppi_edges_genes.tsv")

node_df = pd.read_parquet(node_fp)
expr_cols = node_df.columns[:6]
esm_cols  = node_df.columns[-640:]

# Build (expression + ESM) feature matrix for PCA
feat_df = node_df.loc[:, expr_cols.tolist() + esm_cols.tolist()]

edges_df = pd.read_csv(edge_fp, sep="\t", names=["gene1", "gene2", "combined_score"], header=0)

# ────────────────────────────────────────────────────────────────────────────────
# 2) Build NetworkX graph, compute degree
# ────────────────────────────────────────────────────────────────────────────────
G = nx.Graph()
G.add_nodes_from(node_df.index)

valid_edges = edges_df[
    edges_df["gene1"].isin(node_df.index) &
    edges_df["gene2"].isin(node_df.index)
]
G.add_edges_from(valid_edges[["gene1", "gene2"]].itertuples(index=False, name=None))

deg_series = pd.Series(dict(G.degree()), name="degree")
deg_norm   = (deg_series - deg_series.min()) / (deg_series.max() - deg_series.min())

present_biomarkers = [g for g in known_dmd_genes if g in node_df.index]

# ────────────────────────────────────────────────────────────────────────────────
# 3) 2D Ego Network centered on DMD (high‐quality, professional styling)
# ────────────────────────────────────────────────────────────────────────────────
if "DMD" not in G:
    raise RuntimeError("DMD not found in the PPI graph; cannot plot ego network.")

ego = nx.ego_graph(G, "DMD", radius=1)
pos = nx.spring_layout(ego, seed=42, k=0.3)

# Prepare node attributes
node_colors = []
node_sizes  = []
node_labels = {}
for n in ego.nodes():
    if n == "DMD":
        node_colors.append("firebrick")
        node_sizes.append(300)
        node_labels[n] = n
    elif n in present_biomarkers:
        node_colors.append("darkorange")
        node_sizes.append(200)
        node_labels[n] = n
    else:
        node_colors.append("steelblue")
        # size by degree (scaled)
        node_sizes.append(100 + 200 * (deg_norm[n]))

# Edge widths scaled by combined_score
edge_weights = []
max_score = valid_edges["combined_score"].max()
for u, v in ego.edges():
    score = valid_edges[
        ((valid_edges["gene1"] == u) & (valid_edges["gene2"] == v)) |
        ((valid_edges["gene1"] == v) & (valid_edges["gene2"] == u))
    ]["combined_score"].iloc[0]
    edge_weights.append(0.5 + 2.5 * (score / max_score))

# Plotting
plt.figure(figsize=(8, 8))
sns.set_style("whitegrid")
nx.draw_networkx_edges(
    ego, pos,
    alpha=0.4,
    edge_color="gray",
    width=edge_weights
)
nx.draw_networkx_nodes(
    ego, pos,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="black",
    linewidths=0.6
)
# Only label DMD and known biomarkers
nx.draw_networkx_labels(
    ego, pos,
    labels={n: n for n in node_labels},
    font_size=9,
    font_weight="bold"
)

# Construct custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", label="DMD",
           markerfacecolor="firebrick", markersize=10, markeredgecolor="black"),
    Line2D([0], [0], marker="o", color="w", label="Known biomarkers",
           markerfacecolor="darkorange", markersize=10, markeredgecolor="black"),
    Line2D([0], [0], marker="o", color="w", label="Neighbors",
           markerfacecolor="steelblue", markersize=10, markeredgecolor="black"),
    Line2D([0], [0], color="gray", lw=2, label="Interaction (edge)")
]
plt.legend(handles=legend_elements, loc="best", frameon=True)

plt.title("1-Hop Ego Network Centered on DMD\n(Node size ∝ degree, edge width ∝ score)", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.show()

# ────────────────────────────────────────────────────────────────────────────────
# 4) Degree Distribution Highlighting DMD & Biomarkers
# ────────────────────────────────────────────────────────────────────────────────
plt.figure(figsize=(6, 4))
sns.histplot(deg_series.values, bins=50, color="lightgray", edgecolor="black")
plt.xlabel("Degree", fontsize=12)
plt.ylabel("Number of Genes", fontsize=12)
plt.title("Degree Distribution of DMD Interactome", fontsize=14)

ymax = plt.ylim()[1]
for gene, color in [("DMD", "firebrick")] + [(g, "darkorange") for g in present_biomarkers if g != "DMD"]:
    if gene in deg_series:
        d = deg_series[gene]
        plt.axvline(d, color=color, linestyle="--", linewidth=1.2)
        plt.text(d + 0.5, ymax * 0.9, gene, rotation=90, color=color,
                 fontsize=8, va="top", ha="left")

plt.tight_layout()
plt.show()

# ────────────────────────────────────────────────────────────────────────────────
# 5) PCA of (Expression + ESM) Features, coloring DMD and biomarkers
# ────────────────────────────────────────────────────────────────────────────────
pca = PCA(n_components=2)
X_pca = pca.fit_transform(feat_df.values)
pca_df = pd.DataFrame(X_pca, index=feat_df.index, columns=["PC1", "PC2"])

plt.figure(figsize=(6, 6))
# Plot all genes in light gray
plt.scatter(
    pca_df["PC1"], pca_df["PC2"],
    c="lightgray", s=8, alpha=0.3, label="_nolegend_"
)
# Overlay DMD in red
plt.scatter(
    pca_df.loc["DMD", "PC1"], pca_df.loc["DMD", "PC2"],
    c="firebrick", s=80, edgecolor="black", linewidth=0.8, label="DMD"
)
# Overlay other biomarkers in orange
for gene in present_biomarkers:
    if gene != "DMD" and gene in pca_df.index:
        plt.scatter(
            pca_df.loc[gene, "PC1"],
            pca_df.loc[gene, "PC2"],
            c="darkorange", s=50, edgecolor="black", linewidth=0.6, label=gene
        )

# Avoid duplicate legend entries
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), fontsize=8, loc="best")

plt.xlabel("PC1", fontsize=12)
plt.ylabel("PC2", fontsize=12)
plt.title("PCA of (Expression + ESM) Features\nDMD & Biomarkers Highlighted", fontsize=14)
plt.tight_layout()
plt.show()

# ────────────────────────────────────────────────────────────────────────────────
# 6) Expression Heatmap for Known DMD Biomarkers (polished)
# ────────────────────────────────────────────────────────────────────────────────
expr_only = node_df.loc[present_biomarkers, expr_cols]
expr_logged = np.log2(expr_only + 1)

plt.figure(figsize=(len(expr_cols)*0.6, len(present_biomarkers)*0.5))
sns.heatmap(
    expr_logged,
    cmap="mako",
    center=np.median(expr_logged.values),
    cbar_kws={"label": "log₂(TPM + 1)"},
    yticklabels=True,
    xticklabels=True,
    linewidths=0.5,
    linecolor="white"
)
plt.title("Expression of Known DMD Biomarkers Across Samples", fontsize=14)
plt.ylabel("Gene", fontsize=12)
plt.xlabel("Sample", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


GCN Sequence Data

In [ ]:
import pickle
import pandas as pd
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────────────
# 1) Load the pickle and inspect its contents (“head”)
# ────────────────────────────────────────────────────────────────────────────────
PICKLE_FP = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/all_sequences.pkl")

with open(PICKLE_FP, "rb") as f:
    data = pickle.load(f)

print("Loaded object type:", type(data))
print()

# If it’s a dict, show the first few key→value pairs (truncated)
if isinstance(data, dict):
    print("Dictionary keys (first 10):")
    for i, key in enumerate(data.keys()):
        if i >= 10:
            break
        val = data[key]
        seq_len = len(val) if isinstance(val, str) else "<?>"
        print(f"  {i+1:2d}. {key!r} → sequence length: {seq_len}")
    print("\nExample sequence for one key:")
    sample_key = next(iter(data.keys()))
    print(f"  Key: {sample_key!r}")
    print(f"  Sequence (first 100 aa): {data[sample_key][:100]}…")
    print()

# If it’s a pandas Series, show its head()
elif isinstance(data, pd.Series):
    print("Pandas Series head():")
    display(data.head(10))

    print("\nSeries index name:", data.index.name)
    print("Series dtype:", data.dtype)
    print()

# If it’s a pandas DataFrame, show its head()
elif isinstance(data, pd.DataFrame):
    print("Pandas DataFrame head():")
    display(data.head(10))

    print("\nDataFrame columns:", list(data.columns))
    print("Index name:", data.index.name)
    print("Dtypes:")
    print(data.dtypes)
    print()

else:
    print("→ Unexpected pickle contents. You may need to inspect manually:\n", data)


In [ ]:
# (Only run once. If you already have these libs installed, you can skip.)
!pip install torch torchvision torchaudio torch-geometric matplotlib seaborn biopython


In [ ]:
import pickle
import pandas as pd
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────────────
# 1) Paths
# ────────────────────────────────────────────────────────────────────────────────
PICKLE_FP = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/all_sequences.pkl")
OUT_DIR   = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/processed_dataset")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_TSV   = OUT_DIR / "uniprot_to_sequence.tsv"

print("Loading from pickle:", PICKLE_FP)
print("Will write →", OUT_TSV, "\n")

# ────────────────────────────────────────────────────────────────────────────────
# 2) Load the pickle (a dict like {"1433B_HUMAN": "MXXX…", …})
# ────────────────────────────────────────────────────────────────────────────────
with open(PICKLE_FP, "rb") as f:
    all_seq_dict = pickle.load(f)

print("Loaded object type:", type(all_seq_dict))
if not isinstance(all_seq_dict, dict):
    raise RuntimeError(f"Expected a dict, but got {type(all_seq_dict)}")

# ────────────────────────────────────────────────────────────────────────────────
# 3) Convert to DataFrame
# ────────────────────────────────────────────────────────────────────────────────
rows = []
for full_key, seq in all_seq_dict.items():
    # full_key looks like "1433B_HUMAN" → split off "_HUMAN"
    if full_key.endswith("_HUMAN"):
        uniprot_id = full_key[:-6]  # drop the trailing "_HUMAN"
    else:
        uniprot_id = full_key
    rows.append((uniprot_id, seq))

df = pd.DataFrame(rows, columns=["uniprot_id", "aa_sequence"])
print(f"Total entries in dict: {len(df)}")
print(df.head(10))

# ────────────────────────────────────────────────────────────────────────────────
# 4) Save to TSV
# ────────────────────────────────────────────────────────────────────────────────
df.to_csv(OUT_TSV, sep="\t", index=False)
print(f"\n✔️  Saved {len(df)} rows to:\n   {OUT_TSV}")


In [ ]:
# Cell 1: Build gene_symbol → [UniProt IDs] and UniProt → sequence mappings

import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

# 1a) Load the UniProt → amino-acid sequence TSV
uniprot_seq_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/processed_dataset/uniprot_to_sequence.tsv")
uniprot_df = pd.read_csv(uniprot_seq_fp, sep="\t")  # columns: [uniprot_id, aa_sequence]

# Build a dict for quick lookup: UniProt ID → sequence
uniprot_to_seq = dict(zip(uniprot_df["uniprot_id"], uniprot_df["aa_sequence"]))

# 1b) Read ensembl_gene_to_uniprot.tsv to get UniProt → ENSEMBL mapping
gene_map_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/data/uniprot_mapping/ensembl_gene_to_uniprot.tsv")
gene_map_df = pd.read_csv(
    gene_map_fp,
    sep="\t",
    usecols=["ensembl_gene_id", "uniprot_id"]
)
# Strip version suffix from ENSEMBL IDs (e.g., "ENSG00000141510.16" → "ENSG00000141510")
gene_map_df["ensembl_gene_id"] = gene_map_df["ensembl_gene_id"].astype(str).str.split(".", n=1).str[0]

# 1c) Load GTF to build ENSG → gene_symbol (protein_coding only)
gtf_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/data/gencode.v39.annotation.gtf")
ensembl_to_sym = {}
with open(gtf_fp, "r") as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue
        cols = line.rstrip().split("\t")
        if cols[2] != "gene":
            continue
        attrs = cols[8]
        id_match   = re.search(r'gene_id "([^"]+)"', attrs)
        name_match = re.search(r'gene_name "([^"]+)"', attrs)
        type_match = re.search(r'gene_type "([^"]+)"', attrs)
        if id_match and name_match and type_match:
            ensg_base = id_match.group(1).split(".", 1)[0]
            gene_type = type_match.group(1)
            if gene_type == "protein_coding":
                gene_symbol = name_match.group(1)
                ensembl_to_sym[ensg_base] = gene_symbol

# 1d) Build UniProt_ID → gene_symbol (protein_coding only)
gene_map_nodup = gene_map_df.drop_duplicates(subset="uniprot_id", keep="first")
uniprot_to_symbol = {}
for _, row in gene_map_nodup.iterrows():
    uni = row["uniprot_id"]
    ensg = row["ensembl_gene_id"]
    sym = ensembl_to_sym.get(ensg)
    if sym:
        uniprot_to_symbol[uni] = sym

# 1e) Invert to get gene_symbol → list of UniProt IDs
symbol_to_uniprot = defaultdict(list)
for uni, sym in uniprot_to_symbol.items():
    symbol_to_uniprot[sym].append(uni)

print(f"Built mapping: {len(symbol_to_uniprot)} gene symbols → UniProt ID lists")


In [ ]:
# Cell 2: Load node_features.parquet to get the exact list of gene symbols

import pandas as pd
from pathlib import Path

# If running in Colab, install pyarrow first:
# !pip install pyarrow --quiet

node_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/node_data/node_features.parquet")
node_table = pd.read_parquet(node_fp)
genes_in_nodes = list(node_table.index)
print("Total genes in node_features:", len(genes_in_nodes))


In [ ]:
# Cell 3: For each gene_symbol, retrieve and concatenate all UniProt sequences,
#         then save one row per gene into a TSV.

import pandas as pd
from pathlib import Path

rows = []
for sym in genes_in_nodes:
    uni_list = symbol_to_uniprot.get(sym, [])
    if uni_list:
        # Fetch all sequences for this gene’s mapped UniProt IDs
        seqs = [uniprot_to_seq.get(uni, "") for uni in uni_list]
        # Remove any empty strings (in case a UniProt ID had no entry in the TSV)
        seqs = [s for s in seqs if s]
        # Concatenate multiple isoform sequences with semicolons
        combined_seq = ";".join(seqs)
    else:
        # No UniProt IDs mapped → leave blank
        combined_seq = ""
    rows.append({"gene_symbol": sym, "all_sequences": combined_seq})

final_df = pd.DataFrame(rows)

# Save to a TSV so each gene from node_features has its full set of sequences
output_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/processed_dataset/gene_to_all_sequences.tsv")
final_df.to_csv(output_fp, sep="\t", index=False)
print("Saved gene_to_all_sequences.tsv with", final_df.shape[0], "rows")


In [ ]:
# Cell 4 (revised): Fill in missing sequences for rows where all_sequences is "", whitespace, or NaN

import pandas as pd
import requests
from pathlib import Path
import numpy as np

def fetch_human_protein_sequence(gene_symbol):
    """
    Query UniProt REST for the first reviewed (Swiss-Prot) Homo sapiens entry matching gene_symbol.
    Returns the amino‐acid sequence (no line breaks) or "" if not found.
    """
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    query = f'gene_exact:{gene_symbol} AND organism_id:9606 AND reviewed:true'
    params = {
        "query": query,
        "format": "fasta",
        "fields": "accession,sequence"
    }
    res = requests.get(base_url, params=params, timeout=30)
    if res.status_code != 200 or not res.text.strip():
        return ""
    # Drop header lines (starting with ">") and join the rest
    lines = [line.strip() for line in res.text.splitlines() if not line.startswith(">")]
    return "".join(lines)

# 4a) Load the TSV from Cell 3
input_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/processed_dataset/gene_to_all_sequences.tsv")
df = pd.read_csv(input_fp, sep="\t", dtype={"all_sequences": "string"})

# 4b) Create a mask for genuinely blank entries: NaN, empty, or only whitespace
is_blank = (
    df["all_sequences"].isna()                                              # NaN
    | (df["all_sequences"].astype(str).str.strip() == "")                    # empty or whitespace
)
missing_genes = df.loc[is_blank, "gene_symbol"].tolist()
print(f"Found {len(missing_genes)} genes with blank/NaN sequences → fetching from UniProt...")

# 4c) Loop over each missing gene_symbol and fetch via REST
for gene in missing_genes:
    seq = fetch_human_protein_sequence(gene)
    # Write back into the DataFrame, even if seq is "" (i.e., not found)
    df.loc[df["gene_symbol"] == gene, "all_sequences"] = seq
    if seq:
        print(f"✔ Retrieved {len(seq)} aa for {gene}")
    else:
        print(f"✘ No reviewed sequence found for {gene}")

# 4d) Save to a new TSV (so the original is preserved)
output_fp = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/processed_dataset/gene_to_all_sequences_filled.tsv")
df.to_csv(output_fp, sep="\t", index=False)
print(f"Saved updated TSV with filled sequences to:\n  {output_fp}")


GAT Development

Data Preparation

In [ ]:

!pip install fair-esm biopython requests tqdm --quiet

# Verify installation versions
import esm, Bio, requests, tqdm
print("ESM version:", esm.__version__)
print("Biopython version:", Bio.__version__)
print("Requests version:", requests.__version__)
print("tqdm version:", tqdm.__version__)


In [ ]:
# (Only run once. If you already have these libs installed, you can skip.)
!pip install torch torchvision torchaudio torch-geometric matplotlib seaborn biopython


In [ ]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.utils import dense_to_sparse
from sklearn.model_selection import train_test_split
from Bio import SeqIO

# Paths (adjust if needed)
NODE_FP      = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/node_data/node_features.parquet")
EDGE_FP      = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/ppi_gene_edges/ppi_edges_genes.tsv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "\n")

# A list of known DMD modifier genes (expanded further)
known_dmd_genes = [
    # Core DMD‐pathway genes / biomarkers:
    "DMD", "CKM", "SGCA", "SGCB", "SGCD",
    "CAPN3", "DES", "MYH7", "MYOG", "ACTA1",
    "ACTA2", "DMDP1", "DMDP2",

    # Literature‐validated modifiers (original set):
    "SPP1",    # Osteopontin
    "LTBP4",   # Latent TGF-β Binding Protein 4
    "CD40",
    "ACTN3",   # α-Actinin-3
    "NOS1",    # Neuronal Nitric Oxide Synthase
    "UTRN",    # Utrophin
    "THBS1",   # Thrombospondin-1
    "TGFB1",   # TGF-β1
    "SMAD7",
    "MMP9",
    "COL6A1", "COL6A2", "COL6A3",
    "PLOD1",
    "CDH15",
    "TNF",
    "CCL2",

    # Additional modifiers from broader DMD‐pathway studies:
    "MSTN",      # Myostatin
    "FST",       # Follistatin
    "ANXA6",     # Annexin A6
    "DNM2",      # Dynamin 2
    "XIST",      # X-inactive specific transcript (epigenetic regulator)
    "TGFB2",     # TGF-β2
    "TGFB3",     # TGF-β3
    "COL1A1",    # Collagen type I alpha 1
    "COL3A1",    # Collagen type III alpha 1
    "MMP2",      # Matrix Metalloproteinase-2
    "MMP14",     # Matrix Metalloproteinase-14
    "TIMP1",     # TIMP metallopeptidase inhibitor 1
    "TIMP2",     # TIMP metallopeptidase inhibitor 2
    "EDN1",      # Endothelin-1
    "IL6",       # Interleukin 6
    "IL10",      # Interleukin 10
    "IL1B",      # Interleukin 1 beta
    "TNFRSF1A",  # TNF receptor superfamily member 1A
    "TNFRSF1B",  # TNF receptor superfamily member 1B
    "CD163",     # Macrophage scavenger receptor
    "CD68",      # Macrophage marker
    "ITGB4",     # Integrin beta-4
    "IL4",       # Interleukin 4
    "IL13",      # Interleukin 13
    "CCL5",      # RANTES
    "CCL7",      # MCP-3
    "CCL11",     # Eotaxin
    "CCR2",      # C-C chemokine receptor type 2
    "CCR5",      # C-C chemokine receptor type 5
    "CST3",      # Cystatin C
    "FN1",       # Fibronectin 1

    # Muscle‐regeneration / myogenesis regulators:
    "PAX7",      # Paired box 7
    "MYOD1",     # Myogenic differentiation 1
    "MYF5",      # Myogenic factor 5
    "MYF6",      # Myogenic factor 6

    # Apoptosis / oxidative stress modulators:
    "CASP3",     # Caspase 3
    "CASP8",     # Caspase 8
    "SOD1",      # Superoxide dismutase 1
    "SOD2",      # Superoxide dismutase 2
    "GPX3",      # Glutathione peroxidase 3
    "XRCC1",     # DNA repair protein XRCC1
    "PARP1",     # Poly (ADP‐ribose) polymerase 1

    # Additional extracellular matrix & fibrosis‐related genes:
    "LAMA2",     # Laminin subunit alpha-2
    "LAMA4",     # Laminin subunit alpha-4
    "FNDC5",     # Fibronectin type III domain-containing protein 5
    "ELN",       # Elastin
    "TNC",       # Tenascin C
    "VCAM1",     # Vascular cell adhesion molecule 1
    "ICAM1",     # Intercellular adhesion molecule 1
]


In [ ]:
# -------------------------------------------------------
# Cell 1: Install required libraries (if not already installed)
# -------------------------------------------------------


# -------------------------------------------------------
# Cell 2: Imports and path definitions
# -------------------------------------------------------
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from torch_geometric.utils import remove_self_loops, coalesce

# Adjust these paths if needed
NODE_FP = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/node_data/node_features.parquet")
EDGE_FP = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/ppi_gene_edges/ppi_edges_genes.tsv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------------------------------------
# Cell 3: Load node_features.parquet and inspect
# -------------------------------------------------------
# Read the Parquet into a Pandas DataFrame
node_df = pd.read_parquet(NODE_FP)

# Inspect first few rows and columns
print("Node DataFrame shape:", node_df.shape)
print("Columns sample:", list(node_df.columns[:10]))

# Identify column blocks correctly:
# - First 6 columns are expression
# - Last 640 columns are ESM embeddings
# - Everything in between is GO one-hot
expr_cols = list(node_df.columns[:6])
esm_cols  = list(node_df.columns[-640:])
go_cols   = list(node_df.columns[6:-640])

print("Expression columns:", expr_cols)
print("ESM columns (first few):", esm_cols[:5])
print("GO columns (first few):", go_cols[:5])

# -------------------------------------------------------
# Cell 4: Normalize and preprocess features
# -------------------------------------------------------
# 1. Normalize expression: log1p + z-score
expr_vals   = node_df[expr_cols].values.astype(float)
expr_logged = np.log1p(expr_vals)

expr_mean   = expr_logged.mean(axis=0)
expr_std    = expr_logged.std(axis=0)
expr_zscored = (expr_logged - expr_mean) / (expr_std + 1e-8)

# Replace normalized expression back into DataFrame
node_df[expr_cols] = expr_zscored

# 2. Normalize ESM embeddings: L2-normalize each row
esm_vals = node_df[esm_cols].values.astype(float)
l2_norms = np.linalg.norm(esm_vals, axis=1, keepdims=True) + 1e-8
esm_normalized = esm_vals / l2_norms

# Replace normalized ESM back into DataFrame
node_df[esm_cols] = esm_normalized

# 3. GO one-hot: keep as-is (binary)
#    No transformation needed for go_cols.

# Verify feature matrix creation
X_np = node_df[expr_cols + esm_cols + go_cols].values
print("Combined feature matrix shape:", X_np.shape)

# Convert to torch.Tensor
X_feats = torch.tensor(X_np, dtype=torch.float)
print("X_feats shape (torch):", X_feats.shape)

# -------------------------------------------------------
# Cell 5: Load PPI edges and build edge_index
# -------------------------------------------------------
# Read TSV: assume header row exists
edges_df = pd.read_csv(EDGE_FP, sep="\t", header=0, names=["gene1", "gene2", "combined_score"])
print("Raw edges.csv shape:", edges_df.shape)

# Build gene-to-index mapping from node_df.index
gene2idx = {g: i for i, g in enumerate(node_df.index.tolist())}

# Filter edges where both endpoints appear in node_df
mask = edges_df["gene1"].isin(gene2idx) & edges_df["gene2"].isin(gene2idx)
filtered_edges = edges_df.loc[mask].copy()
print("Filtered edges (both endpoints present):", filtered_edges.shape[0])

# Map gene symbols to integer indices
i_idx = filtered_edges["gene1"].map(gene2idx).values
j_idx = filtered_edges["gene2"].map(gene2idx).values

# Because the graph is undirected, add both (i→j) and (j→i)
all_i = np.concatenate([i_idx, j_idx])
all_j = np.concatenate([j_idx, i_idx])
edge_np = np.stack([all_i, all_j], axis=0)  # shape [2, num_edges*2]

# Convert to torch.LongTensor
edge_index = torch.tensor(edge_np, dtype=torch.long)

# Remove self-loops and coalesce duplicate edges
edge_index, _ = remove_self_loops(edge_index)
edge_index = coalesce(edge_index, None, num_nodes=node_df.shape[0])[0]

print("Final edge_index shape:", edge_index.shape)
print("Number of unique undirected edges:", edge_index.size(1) // 2)

# -------------------------------------------------------
# Cell 6: Summary of prepared data
# -------------------------------------------------------
print("\n=== Data Preparation Complete ===")
print("Number of nodes:", X_feats.size(0))
print("Number of features per node:", X_feats.size(1))
print("Number of edges (undirected):", edge_index.size(1) // 2)

# Move tensors to DEVICE for downstream GAT usage
X_feats = X_feats.to(DEVICE)
edge_index = edge_index.to(DEVICE)


In [ ]:
# -------------------------------------------------------
# Cell 7: Assemble label vector y for known DMD genes
# -------------------------------------------------------

import torch

# List of known DMD genes
known_dmd_genes = [
    "DMD", "CKM", "SGCA", "SGCB", "SGCD",
    "CAPN3", "DES", "MYH7", "MYOG", "ACTA1",
    "ACTA2", "DMDP1", "DMDP2", "SPP1", "LTBP4",
    "CD40", "ACTN3", "NOS1", "UTRN", "THBS1",
    "TGFB1", "SMAD7", "MMP9", "COL6A1", "COL6A2",
    "COL6A3", "PLOD1", "CDH15", "TNF", "CCL2",
    "MSTN", "FST", "ANXA6", "DNM2", "XIST",
    "TGFB2", "TGFB3", "COL1A1", "COL3A1", "MMP2",
    "MMP14", "TIMP1", "TIMP2", "EDN1", "IL6",
    "IL10", "IL1B", "TNFRSF1A", "TNFRSF1B", "CD163",
    "CD68", "ITGB4", "IL4", "IL13", "CCL5",
    "CCL7", "CCL11", "CCR2", "CCR5", "CST3",
    "FN1", "PAX7", "MYOD1", "MYF5", "MYF6",
    "CASP3", "CASP8", "SOD1", "SOD2", "GPX3",
    "XRCC1", "PARP1", "LAMA2", "LAMA4", "FNDC5",
    "ELN", "TNC", "VCAM1", "ICAM1"
]

# Number of genes
num_genes = node_df.shape[0]

# Initialize label tensor with zeros
y = torch.zeros(num_genes, dtype=torch.long)

# Map gene symbol to index using node_df.index
gene2idx = {g: i for i, g in enumerate(node_df.index.tolist())}

# Assign y=1 for any known DMD gene found in gene2idx
for gene in known_dmd_genes:
    idx = gene2idx.get(gene)
    if idx is not None:
        y[idx] = 1

# Print summary
print("Total genes:", num_genes)
print("Number of positives (known DMD genes found):", int(y.sum().item()))

# Move y to DEVICE
y = y.to(DEVICE)


In [ ]:
# -------------------------------------------------------
# Cell 8: Combine original PPI edges with kNN‐derived edges (no SMOTE)
# -------------------------------------------------------

import torch
import numpy as np
from sklearn.neighbors import NearestNeighbors
from torch_geometric.utils import remove_self_loops, coalesce

# 1) Convert features to NumPy for KNN
X_knn_np = X_feats.cpu().numpy()    # (N, F)
num_nodes = X_knn_np.shape[0]

# 2) Build a kNN graph (e.g., k = 10)
k = 10
nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm='auto', metric='euclidean')
nbrs.fit(X_knn_np)
distances, indices = nbrs.kneighbors(X_knn_np, return_distance=True)
# distances.shape = (N, k+1), indices.shape = (N, k+1)

# 3) Create directed kNN edges (drop self at index 0)
rows = np.repeat(np.arange(num_nodes), k)
cols = indices[:, 1 : k + 1].reshape(-1)  # skip the self‐match in column 0
# rows[i] → cols[i] for each i

# 4) Symmetrize to make kNN undirected
edge_i_knn = np.concatenate([rows, cols], axis=0)
edge_j_knn = np.concatenate([cols, rows], axis=0)
edge_knn_np = np.vstack([edge_i_knn, edge_j_knn])  # shape: (2, 2·N·k)

# 5) Convert kNN edges to torch.Tensor and clean up
edge_knn = torch.tensor(edge_knn_np, dtype=torch.long)
edge_knn, _ = remove_self_loops(edge_knn)
edge_knn = coalesce(edge_knn, None, num_nodes=num_nodes)[0]

# 6) Original PPI edges (already on GPU) → bring to CPU for concatenation
edge_ppi = edge_index.cpu()

# 7) Concatenate PPI + kNN, then remove duplicates/self‐loops
merged = torch.cat([edge_ppi, edge_knn.cpu()], dim=1)         # shape: (2, E_ppi + E_knn)
merged, _ = remove_self_loops(merged)
edge_combined = coalesce(merged, None, num_nodes=num_nodes)[0].to(DEVICE)

# 8) Print summary
print("PPI edges (undirected):", edge_ppi.size(1) // 2)
print("kNN edges (undirected):", edge_knn.size(1) // 2)
print("Combined edges (undirected):", edge_combined.size(1) // 2)


In [ ]:
# -------------------------------------------------------
# Cell 8: GraphSMOTE‐style oversampling of DMD modifiers (y == 1)
#           to bring positives to 25% of negatives (no full balance)
# -------------------------------------------------------

import numpy as np
from sklearn.neighbors import NearestNeighbors
from torch_geometric.utils import remove_self_loops, coalesce

# 1) Convert existing data to NumPy for interpolation
X_np    = X_feats.cpu().numpy()     # (N, F)
y_np    = y.cpu().numpy()           # (N,)
edge_np = edge_combined.cpu().numpy()  # (2, E_combined)

# 2) Count original classes and identify minority/majority
unique, counts = np.unique(y_np, return_counts=True)
class_counts = dict(zip(unique, counts))
print("Original class counts:", class_counts)

n_pos      = class_counts.get(1, 0)
n_neg      = class_counts.get(0, 0)
target_pos = int(0.25 * n_neg)               # want positives = 25% of negatives
n_to_generate = max(0, target_pos - n_pos)   # only generate if below target
print(f"Current positives: {n_pos}, target positives: {target_pos}, generating: {n_to_generate}")

if n_to_generate == 0:
    # No augmentation needed; just reuse original tensors
    X_aug_np = X_np.copy()
    y_aug_np = y_np.copy()
    edge_aug_np = edge_np.copy()
else:
    # 3) Get all minority indices
    minority_indices = np.where(y_np == 1)[0]

    # 4) Build adjacency list so synthetic nodes can “inherit” edges
    num_nodes = X_np.shape[0]
    adj_list = [[] for _ in range(num_nodes)]
    for src, dst in zip(edge_np[0], edge_np[1]):
        adj_list[src].append(dst)

    # 5) Fit NearestNeighbors on minority‐class features
    k = min(5, len(minority_indices) - 1)  # up to 5 neighbors
    nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm='auto')
    nbrs.fit(X_np[minority_indices])

    new_features = []
    new_labels   = []
    new_edges    = []

    # 6) Generate synthetic minority nodes
    for _ in range(n_to_generate):
        # a) pick one minority node i at random
        i = np.random.choice(minority_indices)
        x_i = X_np[i].reshape(1, -1)

        # b) find its k nearest minority neighbors (including itself), then drop self
        distances, neighbors = nbrs.kneighbors(x_i, return_distance=True)
        neighbors = neighbors.flatten()
        neighbors = neighbors[neighbors != np.where(minority_indices == i)[0][0]]
        if len(neighbors) == 0:
            j = i
        else:
            j_idx_in_minority = np.random.choice(neighbors)
            j = minority_indices[j_idx_in_minority]

        # c) interpolate feature vector between x_i and x_j
        lam   = np.random.rand()
        x_new = X_np[i] + lam * (X_np[j] - X_np[i])
        new_features.append(x_new)
        new_labels.append(1)  # minority class

        # d) connect synthetic node to all neighbors of i
        new_node_index = num_nodes + len(new_features) - 1
        for nbr in adj_list[i]:
            new_edges.append((new_node_index, nbr))
            new_edges.append((nbr, new_node_index))

    # 7) Stack augmented features + labels + edges
    X_aug_np = np.vstack([X_np, np.array(new_features)])       # (N_res, F)
    y_aug_np = np.concatenate([y_np, np.array(new_labels)])    # (N_res,)

    if len(new_edges) > 0:
        new_edge_np  = np.array(new_edges, dtype=np.int64).T  # (2, #new_edge_pairs)
        edge_aug_np  = np.concatenate([edge_np, new_edge_np], axis=1)
    else:
        edge_aug_np = edge_np.copy()

# 8) Remove self-loops and duplicates via coalesce
edge_aug = torch.tensor(edge_aug_np, dtype=torch.long)
edge_aug, _ = remove_self_loops(edge_aug)
edge_aug = coalesce(edge_aug, None, num_nodes=X_aug_np.shape[0])[0]

# 9) Convert back to torch and send to DEVICE
X_aug_t = torch.from_numpy(X_aug_np).float().to(DEVICE)   # (N_res, F)
y_aug_t = torch.from_numpy(y_aug_np).long().to(DEVICE)    # (N_res,)
edge_aug_t = edge_aug.to(DEVICE)

# 10) Print new class counts
unique_aug, counts_aug = np.unique(y_aug_np, return_counts=True)
print("Augmented class counts:", dict(zip(unique_aug, counts_aug)))
print("X_aug_t shape:", X_aug_t.shape)
print("edge_aug_t shape:", edge_aug_t.shape)


In [ ]:
# -------------------------------------------------------
# Cell 9: Load augmented data from Drive (no SMOTE) and rebuild tensors
# -------------------------------------------------------

import torch
import pandas as pd
import numpy as np
from pathlib import Path
from torch_geometric.utils import remove_self_loops, coalesce

# 1) Define paths to the saved augmented files
BASE_DIR         = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/augmented_data")
PARQUET_FP       = BASE_DIR / "augmented_node_features.parquet"
EDGE_TSV_FP      = BASE_DIR / "augmented_edges.tsv"
TORCH_XY_FP      = BASE_DIR / "augmented_X_y.pt"
TORCH_EDGEIDX_FP = BASE_DIR / "augmented_edge_index.pt"

# 2) Load augmented features and labels from Parquet (optional, for inspection)
df_aug = pd.read_parquet(PARQUET_FP)
print(f"Loaded node DataFrame from:\n  {PARQUET_FP}")
print("First few rows:\n", df_aug.head())

# 3) Alternatively, load X_aug and y_aug directly from the saved PyTorch checkpoint
checkpoint = torch.load(TORCH_XY_FP, map_location=DEVICE)
X_aug_t = checkpoint["X_aug"].to(DEVICE)   # Tensor shape (N_res, F)
y_aug_t = checkpoint["y_aug"].to(DEVICE)   # Tensor shape (N_res,)
print(f"Loaded X_aug and y_aug from:\n  {TORCH_XY_FP}")
print("X_aug_t shape:", X_aug_t.shape, "   y_aug_t shape:", y_aug_t.shape)

# 4) Load augmented edge_index from saved tensor
edge_aug_t = torch.load(TORCH_EDGEIDX_FP, map_location=DEVICE)  # shape [2, E_aug]
edge_aug_t = edge_aug_t.to(DEVICE)
print(f"Loaded edge_aug tensor from:\n  {TORCH_EDGEIDX_FP}")
print("edge_aug_t shape:", edge_aug_t.shape)

# 5) (Optional) Load undirected edge list from TSV for inspection
df_edges = pd.read_csv(EDGE_TSV_FP, sep="\t")
print(f"Loaded edge list DataFrame from:\n  {EDGE_TSV_FP}")
print("First few edges:\n", df_edges.head())

# 6) Verify undirected edge_index consistency (reconstruct edge_index from TSV if needed)
#    If you wish to rebuild from the TSV, uncomment below:
# src = df_edges["node_i"].values
# dst = df_edges["node_j"].values
# # add both directions to get a full edge_index
# edge_full = np.concatenate([src, dst]), np.concatenate([dst, src])  # two arrays of length 2·#undirected
# edge_aug_t = torch.tensor(np.vstack(edge_full), dtype=torch.long).to(DEVICE)
# edge_aug_t, _ = remove_self_loops(edge_aug_t)
# edge_aug_t = coalesce(edge_aug_t, None, num_nodes=X_aug_t.shape[0])[0]
# print("Reconstructed edge_aug_t from TSV, shape:", edge_aug_t.shape)

# 7) (Optional) Summary
num_nodes = X_aug_t.size(0)
num_edges = edge_aug_t.size(1) // 2
print(f"\n=== Augmented graph ready ===")
print(f"Number of nodes: {num_nodes}")
print(f"Number of edges (undirected): {num_edges}")


In [ ]:
# -------------------------------------------------------
# Cell 9: Load augmented data from Drive and create stratified splits
# -------------------------------------------------------

import torch
import numpy as np
from pathlib import Path

# 1) Paths to the saved augmented data on Drive
OUT_DIR = Path("/content/drive/MyDrive/dmd_dmd_graph/proc/augmented_data")
TORCH_NODE_FP = OUT_DIR / "augmented_X_y.pt"
TORCH_EDGE_FP = OUT_DIR / "augmented_edge_index.pt"

# 2) Load augmented features and labels
checkpoint = torch.load(TORCH_NODE_FP, map_location=DEVICE)
X_aug_t = checkpoint["X_aug"].to(DEVICE)   # (N_res, F)
y_aug_t = checkpoint["y_aug"].to(DEVICE)   # (N_res,)

# 3) Load augmented edge_index
edge_aug_t = torch.load(TORCH_EDGE_FP, map_location=DEVICE)  # (2, E_aug)
edge_aug_t = edge_aug_t.to(DEVICE)

# 4) Convert labels to CPU NumPy for splitting
y_aug_np = y_aug_t.cpu().numpy()  # shape: (N_res,)

# 5) Identify positive (minority) and negative indices in augmented set
pos_idx_aug = np.where(y_aug_np == 1)[0]
neg_idx_aug = np.where(y_aug_np == 0)[0]

# 6) Shuffle indices for reproducibility
rng = np.random.default_rng(42)
rng.shuffle(pos_idx_aug)
rng.shuffle(neg_idx_aug)

# 7) Compute split sizes
n_pos_aug = len(pos_idx_aug)
n_neg_aug = len(neg_idx_aug)

# 70% / 15% / 15% for positives
n_pos_train = int(0.70 * n_pos_aug)
n_pos_val   = int(0.15 * n_pos_aug)
n_pos_test  = n_pos_aug - n_pos_train - n_pos_val

p_train_aug = pos_idx_aug[:n_pos_train]
p_val_aug   = pos_idx_aug[n_pos_train : n_pos_train + n_pos_val]
p_test_aug  = pos_idx_aug[n_pos_train + n_pos_val :]

# 70% / 15% / 15% for negatives
n_neg_train = int(0.70 * n_neg_aug)
n_neg_val   = int(0.15 * n_neg_aug)
n_neg_test  = n_neg_aug - n_neg_train - n_neg_val

n_train_aug = neg_idx_aug[:n_neg_train]
n_val_aug   = neg_idx_aug[n_neg_train : n_neg_train + n_neg_val]
n_test_aug  = neg_idx_aug[n_neg_train + n_neg_val :]

# 8) Combine indices into overall train/val/test sets
train_idx_aug = np.concatenate([p_train_aug, n_train_aug])
val_idx_aug   = np.concatenate([p_val_aug,   n_val_aug])
test_idx_aug  = np.concatenate([p_test_aug,  n_test_aug])

# 9) Build boolean masks of length N_res
N_res = y_aug_np.shape[0]
train_mask = torch.zeros(N_res, dtype=torch.bool)
val_mask   = torch.zeros(N_res, dtype=torch.bool)
test_mask  = torch.zeros(N_res, dtype=torch.bool)

train_mask[train_idx_aug] = True
val_mask[val_idx_aug]     = True
test_mask[test_idx_aug]   = True

# 10) Move masks to DEVICE
train_mask = train_mask.to(DEVICE)
val_mask   = val_mask.to(DEVICE)
test_mask  = test_mask.to(DEVICE)

# 11) Print summary
print("Train:   total =", train_mask.sum().item(),
      " positives =", int(y_aug_t[train_mask].sum().item()))
print("Val:     total =", val_mask.sum().item(),
      " positives =", int(y_aug_t[val_mask].sum().item()))
print("Test:    total =", test_mask.sum().item(),
      " positives =", int(y_aug_t[test_mask].sum().item()))


In [ ]:
# -------------------------------------------------------
# Cell 10: Print edge counts for kNN, PPI, and augmented graphs
# -------------------------------------------------------

# Assuming the following tensors are already defined:
#   edge_knn      – torch.LongTensor of shape [2, E_knn] for the kNN graph
#   edge_ppi      – torch.LongTensor of shape [2, E_ppi] for the original PPI graph
#   edge_aug_t    – torch.LongTensor of shape [2, E_aug] for the augmented graph (PPI+kNN with synthetic nodes)

# 1) Compute undirected edge counts
num_knn_edges  = edge_knn.size(1)  // 2
num_ppi_edges  = edge_ppi.size(1)  // 2
num_aug_edges  = edge_aug_t.size(1) // 2

# 2) Print the results
print(f"Number of undirected kNN edges:  {num_knn_edges}")
print(f"Number of undirected PPI edges:  {num_ppi_edges}")
print(f"Number of undirected AUG edges:  {num_aug_edges}")


In [ ]:
# -------------------------------------------------------
# Cell 11: Build PyG Data object with augmented graph
# -------------------------------------------------------

from torch_geometric.data import Data

# Create Data using augmented tensors and masks
data_aug = Data(
    x = X_aug_t,
    edge_index = edge_aug_t,
    y = y_aug_t
)
data_aug.train_mask = train_mask
data_aug.val_mask   = val_mask
data_aug.test_mask  = test_mask

print("Augmented Data summary:")
print("  # nodes:", data_aug.num_nodes)
print("  # edges (undirected):", data_aug.edge_index.size(1) // 2)
print("  Train positives:", int(data_aug.y[data_aug.train_mask].sum().item()),
      "Train total:", int(data_aug.train_mask.sum().item()))
print("  Val   positives:", int(data_aug.y[data_aug.val_mask].sum().item()),
      "Val total:", int(data_aug.val_mask.sum().item()))
print("  Test  positives:", int(data_aug.y[data_aug.test_mask].sum().item()),
      "Test total:", int(data_aug.test_mask.sum().item()))


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import add_self_loops, softmax
from torch.nn import LayerNorm

class EnhancedBayesianGAT(torch.nn.Module):
    def __init__(
        self,
        in_channels,            # raw node feature dimension (e.g., concatenated ESM embeddings + other omics)
        hidden_channels,        # hidden GAT channel size
        out_channels,           # number of classes (e.g., 2 for binary biomarker prediction)
        heads_1=4,
        heads_2=4,
        dropout=0.5,
        use_edge_confidence=True  # whether to add an edge-confidence scalar bias
    ):
        super(EnhancedBayesianGAT, self).__init__()
        self.dropout = dropout
        self.use_edge_confidence = use_edge_confidence

        # -------------------------------------------------------
        # (A) Input MLP fusion: combine pre-computed embeddings with raw features
        # This is a single hidden layer MLP:
        # Input: [N, in_channels] → Hidden: [N, hidden_channels] → Activation →
        #        [N, hidden_channels] ready for GAT layers.
        # If you already have a node-feature matrix x (of size [N, in_channels]),
        # this MLP doesn’t add more than O(N·in·hidden) ops once per forward.
        # -------------------------------------------------------
        self.input_mlp = torch.nn.Sequential(
            torch.nn.Linear(in_channels, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.LayerNorm(hidden_channels)
        )

        # -------------------------------------------------------
        # (B) GATv2 layers with residual + layer normalization
        #   - GATv2Conv: a “dynamic” attention mechanism (GATv2 [1])
        #     that can adapt to changing node-pair patterns.
        #   - Residual: we’ll add the input back to the output when dims match
        #   - LayerNorm: normalize node features per layer
        # -------------------------------------------------------
        # Layer 1: in -> hidden
        self.gat1 = GATv2Conv(
            hidden_channels,
            hidden_channels,
            heads=heads_1,
            dropout=dropout,
            concat=True
        )
        self.ln1 = LayerNorm(hidden_channels * heads_1)

        # Layer 2: hidden*heads_1 -> hidden
        self.gat2 = GATv2Conv(
            hidden_channels * heads_1,
            hidden_channels,
            heads=heads_2,
            dropout=dropout,
            concat=True
        )
        self.ln2 = LayerNorm(hidden_channels * heads_2)

        # Layer 3 (output): hidden*heads_2 -> out_channels
        self.gat3 = GATv2Conv(
            hidden_channels * heads_2,
            out_channels,
            heads=1,
            dropout=dropout,
            concat=False
        )
        self.ln3 = LayerNorm(out_channels)

        # -------------------------------------------------------
        # (C) Edge-confidence bias: if you have a scalar “confidence” per edge (e.g., STRING score),
        # we can add it into the attention coefficients in GATv2Conv. This is done by passing
        # edge_attr=[#edges] and integrating it into the attention computation. If no edge_attr,
        # set use_edge_confidence=False.
        # -------------------------------------------------------

        # -------------------------------------------------------
        # (D) Learnable temperature scalar for final softmax:
        #   - 1 parameter, initialized to 1.0; at inference, we divide logits by T.
        # -------------------------------------------------------
        self.temperature = torch.nn.Parameter(torch.tensor(1.0))

    def forward(self, x, edge_index, edge_confidence=None):
        """
        x: [N, in_channels]
        edge_index: [2, E]
        edge_confidence: (optional) tensor of shape [E] with a scalar confidence (0–1) per edge.
        """
        # ——————————————————————————
        # (1) Fuse input features
        # ——————————————————————————
        h = self.input_mlp(x)  # [N, hidden_channels]

        # ——————————————————————————
        # (2) First GATv2 block
        # ——————————————————————————
        h_in1 = h  # for residual
        h = F.dropout(h, p=self.dropout, training=self.training)
        if self.use_edge_confidence and edge_confidence is not None:
            h1 = self.gat1(h, edge_index, edge_attr=edge_confidence)
        else:
            h1 = self.gat1(h, edge_index)
        h1 = F.elu(h1)
        # residual: only if dims match (hidden*heads_1 == hidden*heads_1)
        if h1.shape[-1] == h_in1.shape[-1]:
            h1 = h1 + h_in1
        h1 = self.ln1(h1)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)

        # ——————————————————————————
        # (3) Second GATv2 block
        # ——————————————————————————
        h_in2 = h1
        if self.use_edge_confidence and edge_confidence is not None:
            h2 = self.gat2(h1, edge_index, edge_attr=edge_confidence)
        else:
            h2 = self.gat2(h1, edge_index)
        h2 = F.elu(h2)
        if h2.shape[-1] == h_in2.shape[-1]:
            h2 = h2 + h_in2
        h2 = self.ln2(h2)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)

        # ——————————————————————————
        # (4) Third (output) GATv2 block
        # ——————————————————————————
        if self.use_edge_confidence and edge_confidence is not None:
            out = self.gat3(h2, edge_index, edge_attr=edge_confidence)
        else:
            out = self.gat3(h2, edge_index)
        # No residual here since out_channels likely ≠ (hidden*heads_2)
        out = self.ln3(out)  # [N, out_channels] raw logits
        return out

    def predict_proba(self, x, edge_index, edge_confidence=None, mc_samples=10):
        """
        MC Dropout + Temperature Scaling:
        — Run mc_samples forward passes with dropout enabled (self.train()),
        — Average softmax( logits / T ) across passes.
        """
        self.train()  # keep dropout active
        all_probs = []
        for _ in range(mc_samples):
            logits = self.forward(x, edge_index, edge_confidence)  # [N, out_channels]
            # apply temperature scaling:
            scaled_logits = logits / self.temperature
            probs = F.softmax(scaled_logits, dim=1)  # [N, out_channels]
            all_probs.append(probs.unsqueeze(0))     # [1, N, out_channels]
        mean_probs = torch.cat(all_probs, dim=0).mean(dim=0)  # [N, out_channels]
        return mean_probs


In [ ]:
# -------------------------------------------------------
# Cell XX (Grid Search): Hyperparameterizing EnhancedBayesianGAT
# -------------------------------------------------------

import torch
import torch.nn.functional as F
import itertools

# Reuse data_aug, DEVICE, and EnhancedBayesianGAT from above

# 1) Define hyperparameter grid
hidden_channels_list = [32, 64, 128]
heads_1_list         = [2, 4]
heads_2_list         = [2, 4]
dropout_list         = [0.3, 0.5]
learning_rate_list   = [0.001, 0.005]
weight_decay_list    = [1e-4, 5e-4]

num_epochs = 100  # you can shorten for grid search, then retrain final on full epochs

# 2) Compute class weights once (doesn’t change across hyperparams)
y_train = data_aug.y[data_aug.train_mask]
n_train_pos = y_train.sum().item()
n_train_neg = (y_train == 0).sum().item()
class_weights = torch.tensor([
    1.0,
    n_train_neg / max(n_train_pos, 1)
], device=DEVICE)

print(f"Class weights (neg, pos) = {class_weights.tolist()}")

# 3) Placeholder to track best overall hyperparam combo
best_overall_f1 = 0.0
best_overall_params = None

# 4) Iterate through all combinations
for hidden_channels, heads_1, heads_2, dropout, lr, wd in itertools.product(
        hidden_channels_list,
        heads_1_list,
        heads_2_list,
        dropout_list,
        learning_rate_list,
        weight_decay_list,
    ):
    # 4a) Instantiate a fresh model for each combination
    model = EnhancedBayesianGAT(
        in_channels=data_aug.num_node_features,
        hidden_channels=hidden_channels,
        out_channels=2,
        heads_1=heads_1,
        heads_2=heads_2,
        dropout=dropout,
        use_edge_confidence=True  # or False if not available
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=wd
    )

    best_val_f1 = 0.0
    best_epoch = 0

    # 4b) Training loop for this hyperparam set
    for epoch in range(1, num_epochs + 1):
        # train()
        model.train()
        optimizer.zero_grad()
        if hasattr(data_aug, "edge_confidence"):
            logits = model(data_aug.x, data_aug.edge_index, data_aug.edge_confidence)
        else:
            logits = model(data_aug.x, data_aug.edge_index, None)
        logits_scaled = logits / model.temperature
        loss = F.cross_entropy(
            logits_scaled[data_aug.train_mask],
            data_aug.y[data_aug.train_mask],
            weight=class_weights
        )
        loss.backward()
        optimizer.step()

        # evaluate on validation
        model.eval()
        with torch.no_grad():
            if hasattr(data_aug, "edge_confidence"):
                logits_val = model(data_aug.x, data_aug.edge_index, data_aug.edge_confidence)
            else:
                logits_val = model(data_aug.x, data_aug.edge_index, None)
            logits_val_scaled = logits_val / model.temperature
            preds = logits_val_scaled[data_aug.val_mask].argmax(dim=1)
            labels = data_aug.y[data_aug.val_mask]
            tp = ((preds == 1) & (labels == 1)).sum().item()
            fp = ((preds == 1) & (labels == 0)).sum().item()
            fn = ((preds == 0) & (labels == 1)).sum().item()
            precision = tp / (tp + fp + 1e-9)
            recall    = tp / (tp + fn + 1e-9)
            val_f1    = 2 * precision * recall / (precision + recall + 1e-9)

        # checkpoint best on validation
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            # Optionally save state dict under a name encoding this combo
            # e.g. torch.save(model.state_dict(), f"best_model_h{hidden_channels}_h1{heads_1}_h2{heads_2}_d{dropout}_lr{lr}_wd{wd}.pth")

        # (Optional) print progress every 20 epochs
        if epoch % 20 == 0 or epoch == 1:
            print(
                f"[HC={hidden_channels}, H1={heads_1}, H2={heads_2}, "
                f"D={dropout:.2f}, LR={lr}, WD={wd}] "
                f"Epoch {epoch:03d} | Loss: {loss:.4f} | Val F1: {val_f1:.4f}"
            )

    # 4c) After training loop, compare to global best
    print(f"→ Combo (HC={hidden_channels}, H1={heads_1}, H2={heads_2}, "
          f"D={dropout:.2f}, LR={lr}, WD={wd}) achieved best Val F1 = {best_val_f1:.4f} at epoch {best_epoch}\n")

    if best_val_f1 > best_overall_f1:
        best_overall_f1 = best_val_f1
        best_overall_params = {
            "hidden_channels": hidden_channels,
            "heads_1": heads_1,
            "heads_2": heads_2,
            "dropout": dropout,
            "learning_rate": lr,
            "weight_decay": wd,
            "best_epoch": best_epoch,
            "best_val_f1": best_val_f1
        }

# 5) Report best hyperparameter combination
print("\n=== Grid Search Complete ===")
print(f"Best validation F1 = {best_overall_f1:.4f}")
print("Best hyperparameters:")
for k, v in best_overall_params.items():
    print(f"  {k}: {v}")


In [ ]:
# -------------------------------------------------------
# Cell XX: Training with Best Hyperparameters (75 Epochs, No Early Stopping)
# -------------------------------------------------------

import torch
import torch.nn.functional as F

# 1) Hyperparameters (from grid search results)
in_channels     = data_aug.num_node_features
hidden_channels = 32       # from grid search
out_channels    = 2        # binary classification
heads_1         = 2        # from grid search
heads_2         = 2        # from grid search
dropout         = 0.5      # from grid search
learning_rate   = 0.005    # from grid search
weight_decay    = 0.0005   # from grid search
num_epochs      = 100       # fixed number of epochs

# 2) Compute class weights on TRAIN split
y_train = data_aug.y[data_aug.train_mask]
n_train_pos = y_train.sum().item()
n_train_neg = (y_train == 0).sum().item()

class_weights = torch.tensor([
    1.0,
    n_train_neg / max(n_train_pos, 1)
], device=DEVICE)

print(f"Class weights (neg, pos) = {class_weights.tolist()}")

# 3) Initialize EnhancedBayesianGAT & optimizer
model = EnhancedBayesianGAT(
    in_channels=in_channels,
    hidden_channels=hidden_channels,
    out_channels=out_channels,
    heads_1=heads_1,
    heads_2=heads_2,
    dropout=dropout,
    use_edge_confidence=True  # or False if not available
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

# 4) Training and evaluation functions
def train():
    model.train()
    optimizer.zero_grad()
    if hasattr(data_aug, "edge_confidence"):
        logits = model(data_aug.x, data_aug.edge_index, data_aug.edge_confidence)
    else:
        logits = model(data_aug.x, data_aug.edge_index, None)

    logits_scaled = logits / model.temperature
    loss = F.cross_entropy(
        logits_scaled[data_aug.train_mask],
        data_aug.y[data_aug.train_mask],
        weight=class_weights
    )
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(mask):
    model.eval()
    if hasattr(data_aug, "edge_confidence"):
        logits = model(data_aug.x, data_aug.edge_index, data_aug.edge_confidence)
    else:
        logits = model(data_aug.x, data_aug.edge_index, None)

    logits_eval = logits / model.temperature
    preds = logits_eval[mask].argmax(dim=1)
    labels = data_aug.y[mask]

    acc = (preds == labels).sum().item() / mask.sum().item()
    tp = ((preds == 1) & (labels == 1)).sum().item()
    fp = ((preds == 1) & (labels == 0)).sum().item()
    fn = ((preds == 0) & (labels == 1)).sum().item()

    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1_score  = 2 * precision * recall / (precision + recall + 1e-9)
    return acc, precision, recall, f1_score

# 5) Training loop with checkpointing (75 epochs, no early stopping)
best_val_f1 = 0.0
best_epoch  = 0

for epoch in range(1, num_epochs + 1):
    loss = train()
    train_acc, train_prec, train_rec, train_f1 = evaluate(data_aug.train_mask)
    val_acc, val_prec, val_rec, val_f1           = evaluate(data_aug.val_mask)

    # Checkpoint if improved
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch
        torch.save(model.state_dict(), "best_enhanced_bayesian_gat.pth")

    # Print status every 10 epochs or first epoch
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | "
              f"Loss: {loss:.4f} | "
              f"Train F1: {train_f1:.4f} | "
              f"Val   F1: {val_f1:.4f}")

print(f"\nBest validation F1: {best_val_f1:.4f} at epoch {best_epoch}")


In [ ]:
import os

proc_dir = "/content/drive/MyDrive/dmd_dmd_graph/proc"
os.makedirs(proc_dir, exist_ok=True)

save_path = os.path.join(proc_dir, "best_enhanced_bayesian_gat.pth")
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")


In [ ]:
# -------------------------------------------------------
# Cell 14: Compute and print all classification metrics on test set (using current model in memory)
# -------------------------------------------------------

import torch
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_recall_fscore_support

model.eval()

# 1) Forward pass to get raw logits on entire dataset (include edge_confidence if available)
if hasattr(data_aug, "edge_confidence"):
    logits = model(data_aug.x, data_aug.edge_index, data_aug.edge_confidence)  # [N_res, 2]
else:
    logits = model(data_aug.x, data_aug.edge_index)  # [N_res, 2]

# 2) Apply learned temperature scaling to logits on test nodes
mask = data_aug.test_mask
logits_test = logits.clone()
logits_test[mask] = logits[mask] / model.temperature  # scale both classes equally

# 3) Predicted labels on test set
preds = logits_test[mask].argmax(dim=1).cpu().numpy()
labels = data_aug.y[mask].cpu().numpy()

# 4) Compute probabilities for ROC AUC: apply softmax with temperature
scaled_logits_all = logits / model.temperature
probs = F.softmax(scaled_logits_all, dim=1)[mask, 1].detach().cpu().numpy()

# 5) Confusion matrix
cm = confusion_matrix(labels, preds)

# 6) Accuracy
accuracy = (preds == labels).mean()

# 7) Precision, Recall, F1 by class and overall
precision, recall, f1, support = precision_recall_fscore_support(
    labels, preds, labels=[0, 1], zero_division=0
)
# precision[0], recall[0], f1[0] → class 0 (negative)
# precision[1], recall[1], f1[1] → class 1 (positive)

# 8) Macro and weighted averages
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    labels, preds, average='macro', zero_division=0
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    labels, preds, average='weighted', zero_division=0
)

# 9) ROC AUC
auc = roc_auc_score(labels, probs)

# 10) Print metrics
print("=== Test Set Classification Metrics ===\n")

print("Confusion Matrix:")
print(cm)
print()

print(f"Accuracy:           {accuracy:.4f}\n")

print("Class-wise Metrics:")
print(f"  Class 0 (Negative):  Precision = {precision[0]:.4f}, Recall = {recall[0]:.4f}, F1 = {f1[0]:.4f}, Support = {support[0]}")
print(f"  Class 1 (Positive):  Precision = {precision[1]:.4f}, Recall = {recall[1]:.4f}, F1 = {f1[1]:.4f}, Support = {support[1]}")
print()

print("Averaged Metrics:")
print(f"  Macro Precision = {precision_macro:.4f}, Macro Recall = {recall_macro:.4f}, Macro F1 = {f1_macro:.4f}")
print(f"  Weighted Precision = {precision_weighted:.4f}, Weighted Recall = {recall_weighted:.4f}, Weighted F1 = {f1_weighted:.4f}")
print()

print(f"ROC AUC (Positive class): {auc:.4f}")


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Cell XX (Bayesian MC‐Dropout): Identify Top 10 Genes → Top Up/Down Regulators
#                                using pre‐saved model, with MC‐dropout for realistic percent‐effects
# ──────────────────────────────────────────────────────────────────────

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy import stats
import random

# ── STEP (0) FIX ALL RANDOM SEEDS ───────────────────────────────────────
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ── STEP (0.5) MOUNT GOOGLE DRIVE (only if not already mounted) ─────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ── STEP (1) RE‐INSTANTIATE YOUR MODEL ARCHITECTURE EXACTLY AS DURING TRAINING ──
in_channels     = data_aug.num_node_features
hidden_channels = 32       # same as your grid‐search “best”
out_channels    = 2        # binary classification
heads_1         = 2        # same as your grid‐search “best”
heads_2         = 2        # same as your grid‐search “best”
dropout         = 0.5      # same as your grid‐search “best”

model = EnhancedBayesianGAT(
    in_channels=in_channels,
    hidden_channels=hidden_channels,
    out_channels=out_channels,
    heads_1=heads_1,
    heads_2=heads_2,
    dropout=dropout,
    use_edge_confidence=True
).to(DEVICE)

# ── STEP (2) LOAD THE SAVED STATE_DICT FROM DRIVE ─────────────────────────
saved_path = "/content/drive/MyDrive/dmd_dmd_graph/proc/best_enhanced_bayesian_gat.pth"
checkpoint = torch.load(saved_path, map_location=DEVICE)
model.load_state_dict(checkpoint)
model.eval()   # dropout off for the single‐pass baseline / attention extraction

# ── STEP (3) RUN MC‐DROPOUT TO GET “mean_probs_all” ───────────────────────
def mc_prob(x_input: torch.Tensor, node_idx: int, mc_samples: int = 50) -> float:
    """
    Returns the MC‐dropout average of P(node=node_idx is positive),
    given feature matrix x_input (already on `DEVICE`). Dropout is enabled inside.
    """
    model.train()   # keep dropout on
    vals = []
    with torch.no_grad():
        for _ in range(mc_samples):
            if hasattr(data_aug, "edge_confidence"):
                logits = model(x_input, data_aug.edge_index, data_aug.edge_confidence)
            else:
                logits = model(x_input, data_aug.edge_index)
            probs = F.softmax(logits / model.temperature, dim=1)
            vals.append(probs[node_idx, 1].unsqueeze(0))
    model.eval()
    return torch.cat(vals, dim=0).mean().item()

# Collect per‐node mean probabilities with MC‐dropout
mc_samples =  fifty = 50
probs_samples = []
with torch.no_grad():
    for _ in range(mc_samples):
        model.train()  # enable dropout at inference
        if hasattr(data_aug, "edge_confidence"):
            logits = model(data_aug.x, data_aug.edge_index, data_aug.edge_confidence)
        else:
            logits = model(data_aug.x, data_aug.edge_index)
        scaled_logits = logits / model.temperature
        probs = F.softmax(scaled_logits, dim=1)  # [N, 2]
        probs_samples.append(probs.unsqueeze(0))  # [1, N, 2]

probs_samples  = torch.cat(probs_samples, dim=0)     # [mc_samples, N, 2]
probs_pos      = probs_samples[:, :, 1]              # [mc_samples, N]
mean_probs_all = probs_pos.mean(dim=0).cpu().numpy()  # [N]
std_probs_all  = probs_pos.std(dim=0).cpu().numpy()   # [N]

del probs_samples
torch.cuda.empty_cache()

# ── STEP (4) PREPARE GENE LIST AND KNOWN‐DMD SET ────────────────────────────
all_genes     = list(node_df.index)
num_original  = len(all_genes)
known_dmd_set = set(known_dmd_genes)

# ── STEP (5) BACKGROUND DISTRIBUTION STATS (NOVEL VS. NON‐DMD) ───────────────
non_dmd_indices = [i for i, g in enumerate(all_genes) if g not in known_dmd_set]
bg_mean = np.mean(mean_probs_all[non_dmd_indices])
bg_std  = np.std(mean_probs_all[non_dmd_indices]) + 1e-9

# ── STEP (6) BUILD “CANDIDATES” EXCLUDING ANY KNOWN‐DMD ─────────────────────
candidates = []
for idx, gene in enumerate(all_genes):
    if gene in known_dmd_set:
        continue
    mean_p = float(mean_probs_all[idx])
    std_p  = float(std_probs_all[idx])
    z_score = (mean_p - bg_mean) / bg_std
    candidates.append((idx, gene, mean_p, std_p, z_score))

# ── STEP (7) SORT BY Z‐SCORE DESCENDING AND TAKE TOP 10 ─────────────────────
candidates_sorted = sorted(candidates, key=lambda x: x[4], reverse=True)
top10 = candidates_sorted[:10]

# ── STEP (8) BUILD A DATAFRAME FOR THE TOP 10 ───────────────────────────────
df_top10 = pd.DataFrame(
    top10,
    columns=['node_index', 'gene', 'mean_prob', 'std_prob', 'z_score']
)
df_top10.insert(0, 'rank', range(1, len(df_top10) + 1))

# ── STEP (9) COMPUTE “NODE DEGREE” FOR EACH TOP 10 CANDIDATE ─────────────────
edge_np = data_aug.edge_index.cpu().numpy()  # [2, E]
for i, row in df_top10.iterrows():
    idx = int(row['node_index'])
    degree = int((edge_np[0] == idx).sum() + (edge_np[1] == idx).sum())
    df_top10.at[i, 'degree'] = degree

# ── STEP (10) EXTRACT FINAL‐LAYER ATTENTION WEIGHTS (NO FILTERING) ─────────
with torch.no_grad():
    x_raw = data_aug.x
    h0 = model.input_mlp(x_raw)  # [N, hidden_channels]

    h0_drop = F.dropout(h0, p=model.dropout, training=False)
    if hasattr(data_aug, "edge_confidence"):
        h1 = model.gat1(h0_drop, data_aug.edge_index, data_aug.edge_confidence)
    else:
        h1 = model.gat1(h0_drop, data_aug.edge_index)
    h1 = F.elu(h1)
    h1 = F.dropout(h1, p=model.dropout, training=False)

    if hasattr(data_aug, "edge_confidence"):
        h2 = model.gat2(h1, data_aug.edge_index, data_aug.edge_confidence)
    else:
        h2 = model.gat2(h1, data_aug.edge_index)
    h2 = F.elu(h2)
    h2 = F.dropout(h2, p=model.dropout, training=False)

    # Move just the final activations to CPU so we can call gat3 with return_attention_weights
    h2_cpu       = h2.cpu()
    edge_idx_cpu = data_aug.edge_index.cpu()
    edge_conf_cpu = (
        data_aug.edge_confidence.cpu() if hasattr(data_aug, "edge_confidence") else None
    )
    del x_raw, h0, h0_drop, h1, h2
    torch.cuda.empty_cache()

    model_cpu = model.to("cpu")
    _, (att_edge_index, att_alpha) = model_cpu.gat3(
        h2_cpu, edge_idx_cpu, edge_attr=edge_conf_cpu, return_attention_weights=True
    )
    att_edge_index = att_edge_index.numpy()    # [2, E_aug]
    att_alpha      = att_alpha.numpy().flatten()  # [E_aug]

    model.to(DEVICE)

# ── STEP (11) BUILD “incoming_attention” MAP: dst_idx → list of (src_idx, weight) ─
incoming_attention = defaultdict(list)
for e_id, w in enumerate(att_alpha):
    src = int(att_edge_index[0][e_id])
    dst = int(att_edge_index[1][e_id])
    if src < num_original and dst < num_original:
        incoming_attention[dst].append((src, float(w)))

# ── STEP (12) HELPER: MC‐DROP OUT “PROB” FOR ONE GENE ──────────────────────────
#             (Dropout ON inside, so multiple stochastic draws.)
def mc_prob_gene(x_input: torch.Tensor, node_idx: int, mc_samples: int = 50) -> float:
    """
    Returns average P(node=node_idx is positive) using MC‐dropout.
    """
    model.train()   # enable dropout
    vals = []
    with torch.no_grad():
        for _ in range(mc_samples):
            if hasattr(data_aug, "edge_confidence"):
                logits = model(x_input, data_aug.edge_index, data_aug.edge_confidence)
            else:
                logits = model(x_input, data_aug.edge_index)
            probs = F.softmax(logits / model.temperature, dim=1)
            vals.append(probs[node_idx, 1].unsqueeze(0))
    model.eval()
    return torch.cat(vals, dim=0).mean().item()

# ── STEP (13) FOR EACH TOP 10 GENE, IDENTIFY GREATEST UP/DOWN REGULATORS BY PERCENT CHANGE ─
regulator_data = []

for _, row in df_top10.iterrows():
    gene = row['gene']
    idx  = int(row['node_index'])

    att_list = incoming_attention.get(idx, [])
    if not att_list:
        # If no incoming edges, skip regulator search
        regulator_data.append({
            'gene':               gene,
            'best_upregulator':   None,
            'up_percent':         0.0,
            'best_downregulator': None,
            'down_percent':       0.0
        })
        continue

    # (A) Compute MC‐dropout baseline probability for this gene
    baseline = mc_prob_gene(data_aug.x, idx, mc_samples=50) + 1e-9

    up_candidates   = []
    down_candidates = []

    # (B) For each incoming neighbor, mask them and recompute MC‐prob
    for src, _ in att_list:
        neigh_gene = all_genes[src]

        # ① Clone x and zero out “src” row (on DEVICE)
        x_masked = data_aug.x.clone()
        x_masked[src] = 0.0

        # ② MC‐prob with that neighbor masked
        masked_prob = mc_prob_gene(x_masked, idx, mc_samples=50)
        delta       = baseline - masked_prob
        pct_change  = (delta / baseline) * 100.0

        if pct_change > 0:
            up_candidates.append((neigh_gene, pct_change))
        elif pct_change < 0:
            down_candidates.append((neigh_gene, pct_change))

    # (C) Pick the largest‐positive (best up) and largest‐negative (best down)
    if up_candidates:
        best_up, up_pct = max(up_candidates, key=lambda x: x[1])
    else:
        best_up, up_pct = (None, 0.0)

    if down_candidates:
        best_down, down_pct = min(down_candidates, key=lambda x: x[1])
    else:
        best_down, down_pct = (None, 0.0)

    regulator_data.append({
        'gene':               gene,
        'best_upregulator':   best_up,
        'up_percent':         up_pct,
        'best_downregulator': best_down,
        'down_percent':       down_pct
    })

# ── STEP (14) BUILD DATAFRAME OF REGULATORS ─────────────────────────────────
df_regulators = pd.DataFrame(regulator_data)
print("\n=== Top Regulators for Each of the Top 10 Genes (MC‐Dropout) ===")
display(df_regulators)

# ── STEP (15) OPTIONAL: VISUALIZE NETWORK FOR THE FIRST GENE IN df_regulators ──
first = df_regulators.iloc[0]
target_gene = first['gene']
target_idx  = all_genes.index(target_gene)

up_reg   = first['best_upregulator']
down_reg = first['best_downregulator']
up_idx   = all_genes.index(up_reg)   if up_reg   is not None else None
down_idx = all_genes.index(down_reg) if down_reg is not None else None

# Build mini‐subgraph containing {target, up_reg, down_reg}
viz_nodes = {target_idx}
if up_idx   is not None: viz_nodes.add(up_idx)
if down_idx is not None: viz_nodes.add(down_idx)

H_sub = nx.Graph()
for n in viz_nodes:
    H_sub.add_node(n)

# Add any PPI edges among those nodes
for src, dst in zip(edge_np[0], edge_np[1]):
    if src in viz_nodes and dst in viz_nodes:
        H_sub.add_edge(int(src), int(dst))

# Fixed‐seed layout
pos = nx.spring_layout(H_sub, seed=42)

node_colors = []
node_sizes  = []
labels      = {}

for n in H_sub.nodes():
    g = all_genes[n]
    labels[n] = g
    if g == target_gene:
        node_colors.append('skyblue')
        node_sizes.append(mean_probs_all[n] * 2000 + 200)
    elif g == up_reg:
        node_colors.append('green')
        node_sizes.append(first['up_percent'] * 10 + 200)  # scale down a bit
    elif g == down_reg:
        node_colors.append('red')
        node_sizes.append(abs(first['down_percent']) * 10 + 200)
    else:
        node_colors.append('lightgray')
        node_sizes.append(150)

plt.figure(figsize=(4, 4))
nx.draw_networkx_edges(H_sub, pos, edge_color='gray', alpha=0.5, width=1.0)
nx.draw_networkx_nodes(H_sub, pos, node_color=node_colors, node_size=node_sizes)
nx.draw_networkx_labels(H_sub, pos, labels, font_size=10)

import matplotlib.patches as mpatches
legend_handles = [
    mpatches.Patch(color='skyblue', label=f'Target: {target_gene}'),
    mpatches.Patch(color='green',   label=f'Upreg: {up_reg} (+{first["up_percent"]:.1f}%)'),
]
if down_reg is not None:
    legend_handles.append(mpatches.Patch(color='red', label=f'Downreg: {down_reg} ({first["down_percent"]:.1f}%)'))

plt.legend(handles=legend_handles, loc='upper right', fontsize=10)
plt.title(f"{target_gene} + Top Regulators (MC‐Dropout)", fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()


POST GAT ANALYSIS

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

# ─────────────────────────────────────────────────────────────────────────────
# STEP (X): COMPUTE RANDOM‐WALK WITH RESTART (RWR) SCORES FOR ALL GENES
# ─────────────────────────────────────────────────────────────────────────────

# (1) Extract the original-edge PPI from data_aug.edge_index
#     We assume 'data_aug', 'all_genes', and 'num_original' are already defined.
edge_np = data_aug.edge_index.cpu().numpy()  # [2, E_aug]

# Keep only edges among original nodes (before augmentation)
mask = (edge_np[0] < num_original) & (edge_np[1] < num_original)
edge_orig = edge_np[:, mask]

# (2) Build an undirected graph G_orig using NetworkX (nodes are integer indices 0..num_original-1)
G_orig = nx.Graph()
G_orig.add_nodes_from(range(num_original))
edges_list = list(zip(edge_orig[0].tolist(), edge_orig[1].tolist()))
G_orig.add_edges_from(edges_list)

print(f"Built original PPI graph: {G_orig.number_of_nodes()} nodes, {G_orig.number_of_edges()} edges.")

# (3) Create adjacency matrix A (dense) for RWR
N = num_original
A = np.zeros((N, N), dtype=float)
for u, v in G_orig.edges():
    A[u, v] = 1.0
    A[v, u] = 1.0  # undirected

# (4) Column-normalize A to get A_norm
col_sums = A.sum(axis=0)
col_sums[col_sums == 0] = 1.0  # avoid division by zero for isolated nodes
A_norm = A / col_sums[np.newaxis, :]

# (5) Build the seed vector 's' from known_dmd_genes
#     We assume 'known_dmd_genes' (list of gene symbols) and 'all_genes' (list of gene symbols) exist.
seed_indices = [all_genes.index(g) for g in known_dmd_genes if g in all_genes]
if len(seed_indices) == 0:
    raise RuntimeError("No known DMD seeds found in 'all_genes' — check gene naming.")

s = np.zeros(N, dtype=float)
for idx in seed_indices:
    s[idx] = 1.0 / len(seed_indices)

# (6) Run RWR iteration until convergence
r = 0.3          # restart probability
p = s.copy()     # initial probability = seed vector
tol = 1e-8
max_iters = 1000

for _ in range(max_iters):
    p_next = (1 - r) * (A_norm @ p) + r * s
    if np.linalg.norm(p_next - p, ord=1) < tol:
        p = p_next
        break
    p = p_next
else:
    print("Warning: RWR did not converge within the maximum iterations.")

# (7) Map RWR scores back to gene symbols and create a ranking DataFrame
rwr_scores = {all_genes[i]: float(p[i]) for i in range(N)}
df_rwr = pd.DataFrame([
    {'gene': gene, 'rwr_score': score}
    for gene, score in rwr_scores.items()
])
df_rwr['rank'] = df_rwr['rwr_score'].rank(method='dense', ascending=False).astype(int)
df_rwr = df_rwr.sort_values('rank').reset_index(drop=True)

# Display the top 20 genes by RWR score
print("\n=== Top 20 Genes by RWR Proximity to Known DMD Seeds ===")
display(df_rwr.head(20))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# (After computing df_rwr as above)
# ─────────────────────────────────────────────────────────────────────────────

# Verify NFATC4’s RWR proximity:
target_gene = 'NFATC4'

if target_gene in df_rwr['gene'].values:
    nf_row = df_rwr.loc[df_rwr['gene'] == target_gene].iloc[0]
    rwr_score = nf_row['rwr_score']
    rank      = nf_row['rank']
    total     = df_rwr.shape[0]
    percentile = 100.0 * (1 - (rank - 1) / total)

    print(f"NFATC4 RWR score       = {rwr_score:.5e}")
    print(f"NFATC4 rank            = {rank} out of {total}")
    print(f"NFATC4 percentile (≈)  = {percentile:.1f}%")
else:
    print("NFATC4 is not present in the graph—cannot compute RWR proximity.")


In [ ]:
target_gene = 'NFATC2'
if target_gene in df_rwr['gene'].values:
    nf2_row = df_rwr.loc[df_rwr['gene'] == target_gene].iloc[0]
    score   = nf2_row['rwr_score']
    rank2   = nf2_row['rank']
    total   = df_rwr.shape[0]
    pct2    = 100 * (1 - (rank2 - 1) / total)
    print(f"NFATC2 RWR score      = {score:.5e}")
    print(f"NFATC2 rank           = {rank2} / {total}")
    print(f"NFATC2 percentile ≈   = {pct2:.1f}%")
else:
    print("NFATC2 not in graph; cannot compute RWR proximity.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: INSTALL AND IMPORT NECESSARY PACKAGES
# ─────────────────────────────────────────────────────────────────────────────
# If you haven’t already installed “mygene”, uncomment the following line:
# !pip install mygene

import pandas as pd
from mygene import MyGeneInfo

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: DEFINE THE GENES OF INTEREST
# ─────────────────────────────────────────────────────────────────────────────
genes_of_interest = ["NFATC4", "NFATC2"]

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: QUERY mygene.info USING querymany TO GET GO ANNOTATIONS
# ─────────────────────────────────────────────────────────────────────────────
mg = MyGeneInfo()

# Use querymany with “scopes='symbol'” to ensure it looks up by gene symbol
gene_info = mg.querymany(
    genes_of_interest,
    scopes="symbol",
    fields="go",
    species="human"
)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: PARSE GO ANNOTATIONS INTO A DATAFRAME
# ─────────────────────────────────────────────────────────────────────────────
records = []
for entry in gene_info:
    gene_symbol = entry.get("query", "")
    go_dict = entry.get("go", {})

    # If no 'go' key or it's empty, skip gracefully
    if not isinstance(go_dict, dict):
        continue

    for ontology in ["BP", "MF", "CC"]:
        terms = go_dict.get(ontology, [])
        if isinstance(terms, dict):
            terms = [terms]
        for term in terms:
            go_id   = term.get("id", "")
            go_name = term.get("term", "")
            records.append({
                "gene":     gene_symbol,
                "ontology": ontology,
                "go_id":    go_id,
                "go_name":  go_name
            })

df_go = pd.DataFrame.from_records(records)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: FILTER FOR DMD-RELEVANT GO TERMS
# ─────────────────────────────────────────────────────────────────────────────
# Define keywords related to DMD pathology/progression:
keywords = [
    "muscle", "myocyte", "myofibril", "sarcolemma",
    "calcium", "regeneration", "apoptosis",
    "inflammation", "fibrosis", "extracellular matrix",
    "cytoskeleton", "cell differentiation", "cell proliferation"
]

# Build a single regex pattern (case-insensitive)
pattern = r"(?i)" + r"|".join(keywords)

# Filter DataFrame where 'go_name' contains any of those keywords
df_go_dmd = df_go[df_go["go_name"].str.contains(pattern, na=False)].copy()

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: DISPLAY FILTERED GO TERMS
# ─────────────────────────────────────────────────────────────────────────────
print("\n=== DMD-Relevant GO Annotations for NFATC4 and NFATC2 ===\n")

if df_go_dmd.empty:
    print("No DMD-relevant GO annotations found for the queried genes.")
else:
    sort_cols = [c for c in ["gene", "ontology", "go_id"] if c in df_go_dmd.columns]
    display(df_go_dmd.sort_values(sort_cols, ignore_index=True))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ASSUMPTIONS (from your prior steps):
#   • all_genes:      list of all gene symbols (length = N)
#   • known_dmd_set:  set of known DMD genes
#   • mean_probs_all: numpy array of shape [N], MC‐dropout mean P(node positive)
#   • std_probs_all:  numpy array of shape [N], MC‐dropout std‐dev
#   • bg_mean, bg_std: floats for background distribution (non‐DMD nodes)
#   • candidates:     list of tuples [(idx, gene, mean_p, std_p, z_score), …]
#                     where idx = integer node index, gene = gene symbol
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd

# (A) Re‐compute “candidates” (non‐DMD genes with z‐scores) if not already stored
candidates = []
for idx, gene in enumerate(all_genes):
    if gene in known_dmd_set:
        continue
    mean_p = float(mean_probs_all[idx])
    std_p  = float(std_probs_all[idx])
    z_score = (mean_p - bg_mean) / bg_std
    candidates.append((idx, gene, mean_p, std_p, z_score))

# (B) Sort candidates by z_score descending
candidates_sorted = sorted(candidates, key=lambda x: x[4], reverse=True)

# (C) Build a DataFrame for easy ranking
df_cand = pd.DataFrame(
    candidates_sorted,
    columns=["node_index", "gene", "mean_prob", "std_prob", "z_score"]
)
df_cand["rank"] = np.arange(1, len(df_cand) + 1)
df_cand["percentile"] = 100.0 * (1 - (df_cand["rank"] - 1) / len(df_cand))

# (D) Define a helper to report a gene’s stats, if present
def report_gene_in_gat(gene_symbol):
    row = df_cand[df_cand["gene"] == gene_symbol]
    if row.empty:
        print(f"{gene_symbol} is either a known‐DMD gene or not present in all_genes.")
    else:
        idx      = int(row["node_index"].iloc[0])
        mean_p   = float(row["mean_prob"].iloc[0])
        std_p    = float(row["std_prob"].iloc[0])
        z_score  = float(row["z_score"].iloc[0])
        rank     = int(row["rank"].iloc[0])
        pct      = float(row["percentile"].iloc[0])

        print(f"\n=== {gene_symbol} (Bayesian GAT Ranking) ===")
        print(f"Node index    : {idx}")
        print(f"Mean P(positive) = {mean_p:.5f}")
        print(f"Std‐dev        = {std_p:.5f}")
        print(f"z‐score        = {z_score:.3f}")
        print(f"Rank           = {rank} / {len(df_cand)}")
        print(f"Percentile     ≈ {pct:.1f}%\n")

# (E) Report for NFATC2 and NFATC4
report_gene_in_gat("NFATC2")
report_gene_in_gat("NFATC4")


In [ ]:
!pip install bioservices

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import networkx as nx
from collections import defaultdict
import random
import requests
import io
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# Assumptions:
#  • `model` is your loaded EnhancedBayesianGAT in eval() mode
#  • `data_aug` is your PyG Data object with attributes x, edge_index, (optionally edge_confidence)
#  • `all_genes` is a Python list of gene‐symbols corresponding to data_aug.x rows
#  • `DEVICE` is set to "cuda" or "cpu"
#  • seed, torch.manual_seed, etc., have been set earlier as in your notebook
# ─────────────────────────────────────────────────────────────────────────────

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# 1) Identify NFATC4 node index
try:
    nfatc4_idx = all_genes.index("NFATC4")
except ValueError:
    raise RuntimeError("NFATC4 not found in all_genes.")

# 2) Extract final‐layer attention weights (as in your STEP 10)
with torch.no_grad():
    x_raw = data_aug.x.to(DEVICE)
    h0 = model.input_mlp(x_raw)
    h0_drop = F.dropout(h0, p=model.dropout, training=False)
    if hasattr(data_aug, "edge_confidence"):
        h1 = model.gat1(h0_drop, data_aug.edge_index.to(DEVICE), data_aug.edge_confidence.to(DEVICE))
    else:
        h1 = model.gat1(h0_drop, data_aug.edge_index.to(DEVICE))
    h1 = F.elu(h1)
    h1_drop = F.dropout(h1, p=model.dropout, training=False)
    if hasattr(data_aug, "edge_confidence"):
        h2 = model.gat2(h1_drop, data_aug.edge_index.to(DEVICE), data_aug.edge_confidence.to(DEVICE))
    else:
        h2 = model.gat2(h1_drop, data_aug.edge_index.to(DEVICE))
    h2 = F.elu(h2)
    h2_drop = F.dropout(h2, p=model.dropout, training=False)

    model_cpu = model.to("cpu")
    h2_cpu = h2_drop.cpu()
    edge_idx_cpu = data_aug.edge_index.cpu()
    edge_conf_cpu = (data_aug.edge_confidence.cpu() if hasattr(data_aug, "edge_confidence") else None)
    _, (att_edge_index, att_alpha) = model_cpu.gat3(
        h2_cpu, edge_idx_cpu, edge_attr=edge_conf_cpu, return_attention_weights=True
    )
    att_edge_index = att_edge_index.numpy()    # [2, E_aug]
    att_alpha      = att_alpha.numpy().flatten()  # [E_aug]
    model.to(DEVICE)

# 3) Build “incoming_attention” map: dst_idx → list of (src_idx, weight)
incoming_attention = defaultdict(list)
for e_id, w in enumerate(att_alpha):
    src = int(att_edge_index[0][e_id])
    dst = int(att_edge_index[1][e_id])
    if src < len(all_genes) and dst < len(all_genes):
        incoming_attention[dst].append((src, float(w)))

# 4) MC‐dropout helper for one node
def mc_prob_gene(x_input: torch.Tensor, node_idx: int, mc_samples: int = 50) -> float:
    model.train()
    vals = []
    with torch.no_grad():
        for _ in range(mc_samples):
            if hasattr(data_aug, "edge_confidence"):
                logits = model(x_input, data_aug.edge_index, data_aug.edge_confidence)
            else:
                logits = model(x_input, data_aug.edge_index)
            probs = F.softmax(logits / model.temperature, dim=1)
            vals.append(probs[node_idx, 1].unsqueeze(0))
    model.eval()
    return torch.cat(vals, dim=0).mean().item()

# 5) Identify top up/down regulators of NFATC4
regulator_list = incoming_attention.get(nfatc4_idx, [])
if not regulator_list:
    raise RuntimeError("NFATC4 has no incoming edges in the graph.")

baseline = mc_prob_gene(data_aug.x, nfatc4_idx, mc_samples=50) + 1e-9

up_candidates, down_candidates = [], []
for src_idx, _ in regulator_list:
    src_gene = all_genes[src_idx]
    x_masked = data_aug.x.clone()
    x_masked[src_idx] = 0.0
    masked_prob = mc_prob_gene(x_masked, nfatc4_idx, mc_samples=50)
    delta = baseline - masked_prob
    pct_change = (delta / baseline) * 100.0
    if pct_change > 0:
        up_candidates.append((src_gene, pct_change))
    elif pct_change < 0:
        down_candidates.append((src_gene, pct_change))

up_candidates.sort(key=lambda x: x[1], reverse=True)
down_candidates.sort(key=lambda x: x[1])

top_up = up_candidates[:5]
top_down = down_candidates[:5]

df_nfatc4_regs = pd.DataFrame({
    "type": ["up"]*len(top_up) + ["down"]*len(top_down),
    "gene": [g for g,_ in top_up] + [g for g,_ in top_down],
    "pct":  [p for _,p in top_up] + [p for _,p in top_down]
})
print("\nTop NFATC4 Regulators (by % effect on P(NFATC4 positive)):\n")
print(df_nfatc4_regs.to_string(index=False))



In [ ]:
import numpy as np

def calml5_effect_once(mc_samples=50):
    baseline2 = mc_prob_gene(data_aug.x,               nfatc2_idx, mc_samples)
    x_masked_calml5 = data_aug.x.clone()
    x_masked_calml5[calml5_idx] = 0.0
    masked  = mc_prob_gene(x_masked_calml5, nfatc2_idx, mc_samples)
    delta   = baseline2 - masked
    return (delta / (baseline2 + 1e-9)) * 100.0

# Run the experiment 10 times (each with 50 MC passes)
effects = [calml5_effect_once(mc_samples=50) for _ in range(10)]
print("Mean effect:    ", np.mean(effects))
print("Std. deviation:", np.std(effects))
